# Soft Sensor - CaO Livre (f-CaO)

Previsao da concentracao de **CaO Livre** no clinquer a partir das variaveis de processo do Forno 1.

3 análises unificadas:

| Escolha | Efeito |
|---|---|
| `LAG_MODE = "mi"` \| `"manual"` | lags por Informacao Mutua **ou** lags fisicos do Infografico Rev.1 |
| `DEPLATO_MODE = "deplato_blip"` \| `"deplato_diff"` | medicoes reais por **plato estavel (sem blips)** **ou** por qualquer mudanca de valor |

Basta ajustar essas opcoes no topo e reexecutar para comparar qualquer combinacao (Exp1..Exp5 + Deep Learning).

## Bibliotecas e imports

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

DIR_DATA = "gs://futurai-jupyter-notebook/CSN Cimentos/" 

## Funções utilitárias

Definições reutilizadas em todo o notebook: avaliação padronizada de métricas (`avaliar_modelo`), plotagem dos resultados (`plotar_resultados`), importância de variáveis e o registro central usado no comparativo final.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Registro central das métricas de cada experimento
resultados_experimentos = []


def plot_subplots(dados, vars_plot1, vars_plot2):
    """Plot interativo (Plotly) com dois painéis empilhados e eixo X compartilhado

    vars_plot1 vão no painel superior; vars_plot2 (ex.: o target) no inferior.
    """
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05)
    for var in vars_plot1:
        fig.add_trace(go.Scatter(x=dados.index, y=dados[var], name=var, mode='lines'), row=1, col=1)
    for var in vars_plot2:
        fig.add_trace(go.Scatter(x=dados.index, y=dados[var], name=var, mode='lines', line_color='black'), row=2, col=1)
    fig.update_layout(
        height=700,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5, title=None),
    )
    return fig


def calcular_metricas(y_true, y_pred):
    """Retorna um dicionário com MAE, RMSE e R² para um par (real, previsto)."""
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }


def avaliar_modelo(nome_modelo, y_train, preds_train, y_test, preds_test, registrar=True):
    """Avalia um experimento nas bases de treino e teste de forma padronizada

    Imprime R²/RMSE/MAE e, se ``registrar=True``, guarda o resultado em
    ``resultados_experimentos`` para o comparativo final. Reexecutar a célula
    sobrescreve o registro anterior de mesmo ``nome_modelo``
    """
    m_train = calcular_metricas(y_train, preds_train)
    m_test = calcular_metricas(y_test, preds_test)

    print(f"=== {nome_modelo} ===")
    print(f"[Treino] R2: {m_train['R2']:7.4f} | RMSE: {m_train['RMSE']:7.4f} | MAE: {m_train['MAE']:7.4f}")
    print(f"[Teste ] R2: {m_test['R2']:7.4f} | RMSE: {m_test['RMSE']:7.4f} | MAE: {m_test['MAE']:7.4f}")

    if registrar:
        resultados_experimentos[:] = [r for r in resultados_experimentos if r['Modelo'] != nome_modelo]
        resultados_experimentos.append({
            'Modelo': nome_modelo,
            'R2 Treino': m_train['R2'], 'RMSE Treino': m_train['RMSE'], 'MAE Treino': m_train['MAE'],
            'R2 Teste': m_test['R2'], 'RMSE Teste': m_test['RMSE'], 'MAE Teste': m_test['MAE'],
        })
    return m_train, m_test


def plotar_resultados(y_train, preds_train, y_test, preds_test, nome_modelo):
    """Plota Real vs Previsto na base de treino (topo) e de teste (base)"""
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharey=True)

    axes[0].plot(y_train.index, np.asarray(y_train).ravel(), label='Real (Laboratório)',
                 marker='o', linestyle='-', color='black', alpha=0.7)
    axes[0].plot(y_train.index, preds_train, label=f'Previsão {nome_modelo}',
                 marker='x', linestyle='--', color='blue', alpha=0.8)
    axes[0].set_title(f'{nome_modelo} - Base de Treino')
    axes[0].set_ylabel('CaO Livre')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)

    axes[1].plot(y_test.index, np.asarray(y_test).ravel(), label='Real (Laboratório)',
                 marker='o', linestyle='-', color='black', alpha=0.7)
    axes[1].plot(y_test.index, preds_test, label=f'Previsão {nome_modelo}',
                 marker='s', linestyle='-.', color='darkorange', alpha=0.8)
    axes[1].set_title(f'{nome_modelo} - Base de Teste')
    axes[1].set_ylabel('CaO Livre')
    axes[1].set_xlabel('Tempo')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()


def tabela_importancias(modelo, feature_names, top=15):
    """Retorna as ``top`` variáveis mais importantes de um modelo baseado em árvores"""
    return (
        pd.DataFrame({'Feature': list(feature_names),
                      'Importancia (%)': modelo.feature_importances_ * 100})
        .sort_values('Importancia (%)', ascending=False)
        .head(top)
        .reset_index(drop=True)
    )


def tabela_resultados():
    """Consolida os experimentos avaliados, ordenados pelo menor RMSE de teste"""
    if not resultados_experimentos:
        print("Nenhum experimento avaliado ainda. Rode as células de modelagem antes")
        return None
    return pd.DataFrame(resultados_experimentos).sort_values('RMSE Teste').reset_index(drop=True)


# === Tratamento de forno parado nos graficos: quebra de linha + sombreado ===
def _intervalos_de_parada(mask):
    """Intervalos contiguos (inicio, fim) em que o forno esteve parado (mask True)."""
    m = mask.fillna(False).astype(bool)
    if not m.any():
        return []
    bloco = (m != m.shift()).cumsum()
    ivs = []
    for _, g in m.groupby(bloco):
        if bool(g.iloc[0]):
            ivs.append((g.index[0], g.index[-1]))
    return ivs

def _sombrear_paradas(ax, t0=None, t1=None):
    """Sombreia no eixo os periodos de forno parado (usa INTERVALOS_PARADA, definido no tratamento de parada)."""
    ivs = globals().get('INTERVALOS_PARADA', [])
    rotulado = False
    for (ini, fim) in ivs:
        if t1 is not None and ini > t1:  continue
        if t0 is not None and fim < t0:  continue
        ax.axvspan(ini, fim, color='0.80', alpha=0.5, lw=0,
                   label=('Forno parado' if not rotulado else None))
        rotulado = True

def _quebrar_paradas(idx_tempo, *series, limite_h=8):
    """Insere NaN entre dois laudos quando ha forno parado (ou gap > limite_h) entre eles,
    evitando a linha artificial. Retorna (idx_novo, *series_quebradas)."""
    idx_tempo = pd.DatetimeIndex(idx_tempo)
    mask = globals().get('MASK_PARADA', None)
    arrs = [np.asarray(s, float).ravel() for s in series]
    xs = []
    outs = [[] for _ in arrs]
    for i in range(len(idx_tempo)):
        if i > 0:
            gap_h = (idx_tempo[i] - idx_tempo[i-1]).total_seconds() / 3600.0
            cortar = gap_h > limite_h
            if not cortar and mask is not None:
                try:    cortar = bool(mask.loc[idx_tempo[i-1]:idx_tempo[i]].any())
                except Exception: cortar = False
            if cortar:
                xs.append(idx_tempo[i-1] + (idx_tempo[i] - idx_tempo[i-1]) / 2)
                for o in outs: o.append(np.nan)
        xs.append(idx_tempo[i])
        for o, a in zip(outs, arrs): o.append(a[i])
    return (pd.DatetimeIndex(xs),) + tuple(np.asarray(o, float) for o in outs)

def plotar_resultados(y_train, preds_train, y_test, preds_test, nome_modelo):
    """Real vs Previsto (treino em cima, teste embaixo). Quebra a linha e sombreia o forno parado."""
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharey=True)

    xt, yr_tr, yp_tr = _quebrar_paradas(y_train.index, y_train, preds_train)
    axes[0].plot(xt, yr_tr, label='Real (Laboratorio)', marker='o', linestyle='-', color='black', alpha=0.7)
    axes[0].plot(xt, yp_tr, label=f'Previsao {nome_modelo}', marker='x', linestyle='--', color='blue', alpha=0.8)
    _sombrear_paradas(axes[0], y_train.index.min(), y_train.index.max())
    axes[0].set_title(f'{nome_modelo} - Base de Treino'); axes[0].set_ylabel('CaO Livre')
    axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.6)

    xe, yr_te, yp_te = _quebrar_paradas(y_test.index, y_test, preds_test)
    axes[1].plot(xe, yr_te, label='Real (Laboratorio)', marker='o', linestyle='-', color='black', alpha=0.7)
    axes[1].plot(xe, yp_te, label=f'Previsao {nome_modelo}', marker='s', linestyle='-.', color='darkorange', alpha=0.8)
    _sombrear_paradas(axes[1], y_test.index.min(), y_test.index.max())
    axes[1].set_title(f'{nome_modelo} - Base de Teste'); axes[1].set_ylabel('CaO Livre'); axes[1].set_xlabel('Tempo')
    axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout(); plt.show()

# Parâmetros

In [ ]:
# ============================================================================
#  CAMINHO DA ANALISE  ->  escolha aqui como rodar os experimentos comparativos
# ============================================================================
# LAG_MODE: como o lag de cada variavel de processo entra no Experimento 1
#   "mi"     -> lag de maior Informacao Mutua (varre 0..180 min)
#   "manual" -> lags fisicos do Infografico Rev.0 (LAGS_INFOGRAFICO, definido na proxima celula)
LAG_MODE = "mi"

# DEPLATO_MODE: como recuperar as medicoes reais do laboratorio a partir do plato
#   "deplato_blip" -> medicao = inicio de cada plato ESTAVEL (>= MIN_PLATO min); exclui os "blips"
#                  de 1 min (valores intermediarios interpolados nas transicoes)
#   "deplato_diff"    -> qualquer mudanca de valor (diff()!=0) conta como medicao (inclui os blips)
DEPLATO_MODE = "deplato_blip"
MIN_PLATO = 10            # (so no modo "deplato_blip") minimo de minutos de um plato estavel

LAG_FALLBACK = "mi"       # (so no modo "deplato_diff") lag p/ variaveis fora do dict: "mi" ou 0
# ============================================================================

base_name = "CaO Livre.csv"
timestamp = "Timestamp"
process_name = "CaO Livre"
company_name = "CSN Cimentos"
subsistema = DIR_DATA + process_name + '_subsistema.csv'
target_col = 'AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS'

print(f"CAMINHO -> LAG_MODE='{LAG_MODE}' | DEPLATO_MODE='{DEPLATO_MODE}'"
      + (f" (MIN_PLATO={MIN_PLATO} min)" if DEPLATO_MODE == 'deplato_blip' else ""))

#### Hiperparâmetro de lags por variável (Experimento 1)
LAG_MODE controla como o lag de cada variavel de processo e definido no Exp1:
- "mi"     -> lag de maior Informação Mútua (comportamento padrao; varre 0...max_lag)
- "manual" -> usa LAGS_POR_VARIAVEL onde houver; nas demais, cai em LAG_FALLBACK
- LAG_FALLBACK (so no modo "manual"): "mi" (usa o lag de MI) ou 0 (lag contemporaneo)
- LAGS_POR_VARIAVEL: dict {tag_da_variavel: lag_em_minutos}. Vazio = tudo por MI. Ex: {'AR.F1.FI2719': 117, 'AR.F1.II12501': 123, 'AR.F1.SI12501': 157, 'AR.F1.TIRJ2501': 84}

In [ ]:
# --- Matriz de lags FISICA (Infografico "Timing de producao de clinquer" - Rev.0) ---
# Usada quando LAG_MODE == "manual". lag(tag) = LAB_DELAY_MIN + transporte(etapa da tag).
LAB_DELAY_MIN = 55  # coleta do clinquer -> lancamento no LIMS/PI
_TRANSPORTE_ETAPA = {   # min da etapa ate a coleta do clinquer (marco F)
    'alimentacao':  50, 'torre': 45, 'entrada': 45, 'calcinacao': 30,
    'queima': 20, 'tiragem': 20, 'resfriamento': 5,
}
_ETAPA_TAG = {
    'alimentacao': ['AR.F1.SAIDA_SILO_FARINHA_FSC_LIMS', 'AR.F1.MFARINHA_FSC_LIMS',
                    'AR.F1.MFARINHA_MS_LIMS', 'AR.F1.SAIDA_SILO_FARINHA_MS_LIMS',
                    'AR.F1.MFARINHA_MA_LIMS', 'AR.F1.SAIDA_SILO_FARINHA_MA_LIMS',
                    'AR.F1.MFARINHA_RET#170_LIMS', 'AR.F1.FARINHA_PROD_MGO_SMOINHO_LIMS',
                    'AR.F1.FI1912SP', 'AR.F1.LCB2365_IN_FEEDRATE'],
    'torre':       ['AR.F1.TI12406', 'AR.F1.TI152406', 'AR.F1.TI162406'],
    'entrada':     ['AR.F1.PI2501', 'AR.F1.AI52406', 'AR.F1.AI32406', 'AR.F1.AI42406'],
    'calcinacao':  ['AR.F1.FI2716', 'AR.F1.CONS_CALOR', 'AR.F1.WI2715',
                    'AR.F1.ULT_EST_FARINHA_GRAU_DESCARB_LIMS'],
    'queima':      ['AR.F1.TIRJ2501', 'AR.F1.FI2719', 'AR.F1.TI192501', 'AR.F1.II12501',
                    'AR.F1.SI12501', 'AR.F1.GRAU_ENCH', 'AR.F1.WI2718', 'AR.F1.MCOQUE_PCI_LIMS',
                    'AR.F1.MIXCOMB_CINZAS_MCOMB1_LIMS', 'AR.F1.CLINQUER_FL-1338_SRESF_LIMS'],
    'tiragem':     ['AR.F1.SI12201_SP', 'AR.F1.SI2623'],
    'resfriamento':['AR.F1.PI12606', 'AR.F1.PI22606', 'AR.F1.PI32606', 'AR.F1.PI42606',
                    'AR.F1.PI52606', 'AR.F1.PI62606', 'AR.F1.TI2628_FLTR'],
}
LAGS_INFOGRAFICO = {tag: LAB_DELAY_MIN + _TRANSPORTE_ETAPA[etapa]
                    for etapa, tags in _ETAPA_TAG.items() for tag in tags}

# Seleciona o dicionario de lags conforme o CAMINHO escolhido no topo
LAGS_POR_VARIAVEL = LAGS_INFOGRAFICO if LAG_MODE == "manual" else {}
print(f"LAGS: modo '{LAG_MODE}' -> {len(LAGS_POR_VARIAVEL)} variaveis com lag manual "
      f"({'infografico Rev.1' if LAG_MODE == 'manual' else 'todas por Informacao Mutua'})")

# Carregamento dos dados

Leitura do CSV, reamostragem para 1 minuto e indexação temporal.

In [ ]:
df_dataset = pd.read_csv(DIR_DATA + base_name, sep=";", decimal=".", encoding="utf-8-sig")
df_dataset[timestamp] = pd.to_datetime(df_dataset[timestamp], format="%Y-%m-%d %H:%M:%S")
df_dataset = df_dataset.set_index(timestamp)
df_dataset = df_dataset.resample('1min').asfreq()
df_dataset = df_dataset.reset_index()
df_dataset.head()

In [ ]:
for col in df_dataset.columns:
    if col != 'Timestamp':
        df_dataset[col] = pd.to_numeric(df_dataset[col], errors='coerce')
df_dataset

## TAGs do subsistema CaO Livre

In [ ]:
df_sistema = pd.read_csv(subsistema, sep=";")
df_sistema

# Seleção do subconjunto

Recorte do período com amostragem consistente de 1 minuto.

In [ ]:
start_date = pd.to_datetime("2026-01-01 00:00:00")
end_date = pd.to_datetime("2026-08-01 00:00:00")

mask = (df_dataset[timestamp] >= start_date) & (df_dataset[timestamp] <= end_date)
df_dataset = df_dataset.loc[mask]
df_dataset.set_index(timestamp, inplace=True)
df_dataset

# Análise Exploratória (EDA)

## Informações básicas do dataset

In [ ]:
# Dimensões do Dataset
print(df_dataset.shape)

In [ ]:
# Informações das Colunas
df_dataset.info()

## Verificação de valores ausentes

In [ ]:
# Verificação de NaN
missing_data = df_dataset.isnull().sum()
missing_percent = (missing_data / len(df_dataset)) * 100

df_missing = pd.DataFrame({'Valores Ausentes': missing_data, '% Ausente': missing_percent})
df_missing.sort_values(by='% Ausente', ascending=False, inplace=True)
df_missing.head()

In [ ]:
# Removendo tags com muitos valores faltantes
df_dataset.drop([], axis=1, inplace=True, errors='ignore')
df_dataset.head()

## Variáveis de baixa variância

Identifica constantes e setpoints (baixo coeficiente de variação) candidatos a descarte.

In [ ]:
# Análise de Variância (Filtro de Constantes e Setpoints)

# Isola as variáveis preditoras
features = [col for col in df_dataset.columns if col not in [target_col]]

# Calcula o Desvio Padrão e a Média
df_var = pd.DataFrame({
    'Desvio Padrão': df_dataset[features].std(),
    'Média': df_dataset[features].mean()
})

# Calcula o Coeficiente de Variação (CV) em %: (Std / |Mean|) * 100
# Evita divisão por zero preenchendo com NaN onde a média for exatamente zero
df_var['CV (%)'] = (df_var['Desvio Padrão'] / df_var['Média'].replace(0, np.nan).abs()) * 100

# Ordena das variáveis MENOS variáveis para as mais variáveis
df_var = df_var.sort_values(by='CV (%)', ascending=True)

# Exibe as 20 variáveis mais estáveis (menor CV)
print("Top 20 variáveis mais estáveis (Candidatas a descarte):")
display(df_var.head(20).style.format("{:.4f}").background_gradient(cmap='Reds_r', subset=['CV (%)']))

In [ ]:
# Removendo tags com baixa variação
df_dataset.drop([], axis=1, inplace=True, errors='ignore') # Nenhuma TAG removida devido baixa variação
df_dataset.head()

In [ ]:
df_dataset.ffill(inplace=True)

## Retirada de desligados do forno

In [ ]:
print(f"Total de linhas antes do tratamento de parada: {len(df_dataset)}")

# Identifica os momentos em que o forno está parado (Corrente/Velocidade < 200)
mask_parada = df_dataset['AR.F1.II12501'] < 200

# Conta quantos minutos o forno ficou parado
minutos_parados = mask_parada.sum()
print(f"Minutos com o forno parado (< 200): {minutos_parados}")

# Substitui todos os dados de processo por NaN durante a parada isso garante que qualquer cálculo de Lag ou Janela (rolling) que passe por esse período resulte em NaN e seja posteriormente descartado pelo dropna()
features_para_limpar = [col for col in df_dataset.columns if col not in [timestamp]]
df_dataset.loc[mask_parada, features_para_limpar] = np.nan

print("Valores de parada substituídos por NaN para quebrar o histórico de features.")

# Guarda info de parada para os graficos (sombreado + quebra de linha)
MASK_PARADA = mask_parada.copy()
INTERVALOS_PARADA = _intervalos_de_parada(mask_parada)
print(f'Intervalos de parada detectados: {len(INTERVALOS_PARADA)}')

## Análise da variável target

In [ ]:
# Estatísticas básicas e distribuição da medida de laboratório
display(df_dataset.describe())

plt.figure(figsize=(10, 5))
sns.histplot(df_dataset[target_col].dropna(), kde=False, bins=30, color='blue')
plt.title('Distribuição da Variável Target (Medidas de Laboratório)')
plt.xlabel(target_col)
plt.ylabel('Frequência')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Intervalo de amostragem do laboratório

In [ ]:
# Intervalo de amostragem do laboratorio (depende de DEPLATO_MODE).
_s = df_dataset[target_col]
if DEPLATO_MODE == "deplato_blip":
    _run_id = _s.ne(_s.shift()).cumsum()
    _run_len = _s.groupby(_run_id).transform('size')
    _eh = (_run_len >= MIN_PLATO) & _s.notna() & (_run_id != _run_id.shift())
    instantes_reais = _s.index[_eh].to_series()
else:  # deplato_diff: qualquer mudanca de valor
    _alvos = _s.dropna()
    _mask = _alvos.diff() != 0; _mask.iloc[0] = True
    instantes_reais = _alvos[_mask].index.to_series()

intervalos_minutos = (instantes_reais.diff().dt.total_seconds() / 60).dropna()
print(f"--- Intervalo de Medicao do Laboratorio (DEPLATO_MODE='{DEPLATO_MODE}') ---")
print(f"Medicoes: {len(instantes_reais)} | Media: {intervalos_minutos.mean():.1f} min | "
      f"Mediana: {intervalos_minutos.median():.1f} min | Min: {intervalos_minutos.min():.1f} | Max: {intervalos_minutos.max():.1f}")

plt.figure(figsize=(10, 5))
intervalos_minutos.plot(kind='hist', bins=50, color='#1f77b4', edgecolor='black')
plt.title(f"Intervalos de Medicao (DEPLATO_MODE='{DEPLATO_MODE}') - {target_col}")
plt.xlabel('Minutos entre Medicoes'); plt.ylabel('Frequencia')
plt.axvline(intervalos_minutos.median(), color='red', linestyle='--',
            label=f'Mediana ({intervalos_minutos.median():.0f} min)')
plt.legend(); plt.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()


### Recuperação das medições reais

Desfaz o platô (forward-fill) para isolar os instantes reais de medição do laboratório.

In [ ]:
# Recupera os instantes REAIS de medicao do laboratorio (depende de DEPLATO_MODE).
# target_col ja esta NaN durante paradas de forno (celula anterior).
_s = df_dataset[target_col]
target_limpo = target_col + '_CLEAN'
if DEPLATO_MODE == "deplato_blip":
    # medicao = INICIO de cada plato ESTAVEL (>= MIN_PLATO min); exclui blips de 1 min;
    # preserva medicoes reais iguais adjacentes separadas por um blip.
    _run_id = _s.ne(_s.shift()).cumsum()
    _run_len = _s.groupby(_run_id).transform('size')
    _eh_medicao = (_run_len >= MIN_PLATO) & _s.notna() & (_run_id != _run_id.shift())
else:
    # deplato_diff: qualquer mudanca de valor conta como medicao (inclui os blips)
    _eh_medicao = (_s.diff() != 0)
    _eh_medicao.iloc[0] = True
df_dataset[target_limpo] = _s.where(_eh_medicao, np.nan)

print(f"Total de linhas: {len(df_dataset)}")
print(f"[DEPLATO_MODE='{DEPLATO_MODE}'] medicoes reais resgatadas: {df_dataset[target_limpo].notnull().sum()}")

df_plot = df_dataset[target_limpo].dropna()
fig, ax = plt.subplots(figsize=(15, 4))
xx, yy = _quebrar_paradas(df_plot.index, df_plot.values)
ax.plot(xx, yy, linestyle='-', color='black', linewidth=1.5, label='CaO Livre (medicoes reais)')
_sombrear_paradas(ax, df_plot.index.min(), df_plot.index.max())
ax.set_title(f"CaO Livre - medicoes reais (DEPLATO_MODE='{DEPLATO_MODE}')")
ax.set_xlabel('Tempo'); ax.set_ylabel(target_col)
ax.legend(); ax.grid(True, linestyle='--', alpha=0.6); plt.show()


## Informação Mútua com lags

Varre lags de 0 a 180 min e mantém, por variável, o lag de maior informação mútua.

In [ ]:
# Tabela ordenada (Top Informação Mútua)
from sklearn.feature_selection import mutual_info_regression
import numpy as np
import pandas as pd

max_lag = 180

# Isola as variáveis preditoras
features = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]

# 1. Trata NaNs nas features contínuas para o cálculo não quebrar
df_features_filled = df_dataset[features].ffill().bfill()

# --- DEFINIÇÃO DO CORTE TREINO/TESTE (evita vazamento na seleção de lags) ---
# A seleção de lags por Informação Mútua NÃO pode enxergar o período de teste.
# Definido aqui a data de corte (80% das medições de laboratório) e reutiliada
# no split do modelo, garantindo coerência e ausência de look-ahead.
_instantes_lab = df_dataset[target_limpo].dropna().index
_cut = max(1, int(len(_instantes_lab) * 0.8))
train_cutoff_date = _instantes_lab[_cut - 1]
print(f'Data de corte treino/teste: {train_cutoff_date} '
      f'({_cut} medições de treino / {len(_instantes_lab) - _cut} de teste)')

# Isola apenas os instantes de tempo onde existe laudo real do laboratório
y_target = df_dataset[target_limpo]
indices_validos = y_target.dropna().index
indices_validos = indices_validos[indices_validos <= train_cutoff_date]  # SOMENTE TREINO
y_valid = y_target.loc[indices_validos]

# Preparação da matriz
mi_matrix = np.zeros((len(features), max_lag + 1))

print("Calculando Informação Mútua por lag (isso pode levar alguns minutos)...")

# Cálculo da Informação Mútua por lag
for lag in range(max_lag + 1):
    # Desloca as features no tempo
    shifted_features = df_features_filled.shift(lag)
    
    # Filtra as features deslocadas para coincidir com os instantes do laudo
    X_valid = shifted_features.loc[indices_validos]
    
    # O shift pode gerar NaNs nas primeiras linhas; tratamos com backward fill
    X_valid = X_valid.bfill().fillna(0) 
    
    # Calcula a MI para todas as features de uma vez contra o target contínuo
    mi_scores = mutual_info_regression(X_valid, y_valid, random_state=42)
    mi_matrix[:, lag] = mi_scores

# Converte para DataFrame
df_lags_mi = pd.DataFrame(mi_matrix, index=features, columns=range(max_lag + 1))

# Transformação e ordenação para a tabela
df_tabela_mi = df_lags_mi.unstack().reset_index()
df_tabela_mi.columns = ['Lag (min)', 'Variável', 'Informação Mútua']
df_tabela_mi = df_tabela_mi.sort_values(by='Informação Mútua', ascending=False)

# Mantém apenas a primeira ocorrência (o lag de maior MI) de cada variável
df_tabela_mi = df_tabela_mi.drop_duplicates(subset=['Variável'], keep='first').reset_index(drop=True)

# Exibição do resultado
display(df_tabela_mi.head(20).style.background_gradient(cmap='viridis', subset=['Informação Mútua']))

In [ ]:
# === Hiperparametro de lags por variavel: aplica o override sobre df_tabela_mi ===
# df_tabela_mi e a fonte unica de lags do Exp1 (usada tanto na construcao do dataset
# de features quanto no walk-forward), entao aplicar o override aqui propaga para os
# dois caminhos automaticamente. Reexecucao e idempotente: sempre parte dos lags de MI.
_col_var = 'Variável' if 'Variável' in df_tabela_mi.columns else df_tabela_mi.columns[1]
_col_lag = 'Lag (min)'

# Preserva os lags de Informacao Mutua (referencia/comparacao)
if 'df_tabela_mi_mi' not in globals():
    df_tabela_mi_mi = df_tabela_mi.copy()
df_tabela_mi = df_tabela_mi_mi.copy()

if LAG_MODE == 'manual':
    def _resolver_lag(row):
        var = row[_col_var]
        if var in LAGS_POR_VARIAVEL:
            return int(LAGS_POR_VARIAVEL[var])
        return 0 if LAG_FALLBACK == 0 else int(row[_col_lag])
    df_tabela_mi[_col_lag] = df_tabela_mi.apply(_resolver_lag, axis=1)
    _n_manual = int(df_tabela_mi[_col_var].isin(LAGS_POR_VARIAVEL).sum())
    print(f"LAG_MODE='manual': {_n_manual} variavel(is) com lag manual | "
          f"{len(df_tabela_mi) - _n_manual} por fallback '{LAG_FALLBACK}'.")
    display(df_tabela_mi[[_col_var, _col_lag]].head(20))
elif LAG_MODE == 'mi':
    print("LAG_MODE='mi': lags por Informacao Mutua (sem override).")
else:
    raise ValueError(f"LAG_MODE invalido: {LAG_MODE!r} (use 'mi' ou 'manual').")

## Ferramentas de avaliação padronizada (walk-forward)

Para uma comparacao **justa**, cada experimento e avaliado no **mesmo protocolo**: mesmas 5 janelas walk-forward (expansivas, a partir de 50% da serie), mesmas metricas (R2, RMSE, MAE) e mesmo grafico **Real x Previsto**

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, ElasticNetCV, LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import xgboost as xgb
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAS_SM = True
except Exception:
    _HAS_SM = False

_serie_wf = df_dataset[target_limpo].dropna()
inst_wf   = _serie_wf.index
y_wf      = _serie_wf.reset_index(drop=True)
N_wf      = len(y_wf)
ylag1_wf  = y_wf.shift(1)
ylag2_wf  = y_wf.shift(2)

N_FOLDS_WF, FRAC0_WF = 5, 0.5
_ini = int(N_wf * FRAC0_WF); _step = (N_wf - _ini) // N_FOLDS_WF
FOLDS_WF = [(_ini + f*_step, _ini + (f+1)*_step if f < N_FOLDS_WF-1 else N_wf) for f in range(N_FOLDS_WF)]
TESTE_INI_WF = _ini

def _r2(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return 1 - np.sum((a-b)**2) / np.sum((a-a.mean())**2)
def _rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a,float)-np.asarray(b,float))**2)))
def _mae(a, b):  return float(np.mean(np.abs(np.asarray(a,float)-np.asarray(b,float))))

RESULTADOS_WF = []
PREDICOES_WF  = {}

def avaliar_wf(secao, nome, fitpred, verbose=True):
    """Roda o walk-forward de um modelo (fitpred(a,b)->previsoes), registra e (verbose) imprime as metricas."""
    r2s, rmses, maes, preds = [], [], [], []
    for (a, b) in FOLDS_WF:
        try:
            p = np.asarray(fitpred(a, b), float)
            if p.shape[0] != (b-a) or not np.all(np.isfinite(p)): raise ValueError()
        except Exception:
            p = np.full(b-a, float(y_wf.iloc[1:a].mean()))
        yt = y_wf.iloc[a:b].values
        r2s.append(_r2(yt,p)); rmses.append(_rmse(yt,p)); maes.append(_mae(yt,p)); preds.extend(p.tolist())
    preds = np.array(preds)
    m = {'Secao': secao, 'Modelo': nome, 'R2 (médio)': float(np.mean(r2s)),
         'R2 (global)': _r2(y_wf.iloc[TESTE_INI_WF:N_wf].values, preds),
         'RMSE (médio)': float(np.mean(rmses)), 'MAE (médio)': float(np.mean(maes))}
    RESULTADOS_WF[:] = [r for r in RESULTADOS_WF if r['Modelo'] != nome]
    RESULTADOS_WF.append(m)
    PREDICOES_WF[nome] = (inst_wf[TESTE_INI_WF:N_wf], y_wf.iloc[TESTE_INI_WF:N_wf].values, preds)
    if verbose:
        print(f"=== {nome} (walk-forward, {N_FOLDS_WF} janelas) ===")
        print(f"R2 médio: {m['R2 (médio)']:7.4f} | R2 global: {m['R2 (global)']:7.4f} | "
              f"RMSE: {m['RMSE (médio)']:7.4f} | MAE: {m['MAE (médio)']:7.4f}")
    return m

def tabela_wf(secao=None):
    df = pd.DataFrame(RESULTADOS_WF)
    if secao is not None and len(df): df = df[df['Secao'] == secao]
    return df.sort_values('R2 (médio)', ascending=False).reset_index(drop=True)

def plotar_real_previsto(nome, sufixo=''):
    """Real vs Previsto (walk-forward, out-of-sample), no MESMO estilo do plotar_resultados:
    real preto 'o', previsto laranja 's', com quebra de linha e sombreado de forno parado."""
    inst_oos, y_oos, p_oos = PREDICOES_WF[nome]
    r = next(x for x in RESULTADOS_WF if x['Modelo'] == nome)
    fig, ax = plt.subplots(figsize=(16, 5))
    xx, yr, yp = _quebrar_paradas(inst_oos, y_oos, p_oos)
    ax.plot(xx, yr, label='Real (Laboratorio)', marker='o', linestyle='-', color='black', alpha=0.7)
    ax.plot(xx, yp, label=f'Previsao {nome}', marker='s', linestyle='-.', color='darkorange', alpha=0.8)
    _sombrear_paradas(ax, pd.Timestamp(inst_oos[0]), pd.Timestamp(inst_oos[-1]))
    ax.set_title(f"{nome}{sufixo} - Walk-forward  |  R2={r['R2 (médio)']:.3f}  RMSE={r['RMSE (médio)']:.3f}")
    ax.set_ylabel('CaO Livre'); ax.set_xlabel('Tempo')
    ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout(); plt.show()

def avaliar_secao(secao, modelos, plot_nome=None):
    for nome, fn in modelos.items():
        avaliar_wf(secao, nome, fn, verbose=False)
    df = tabela_wf(secao)
    display(df.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df.columns if c not in ['Secao','Modelo']})
            .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
            .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
    alvo = plot_nome or df.iloc[0]['Modelo']
    plotar_real_previsto(alvo, sufixo=f'  [melhor de {secao}]')
    return df

KSEL_WF = 50
def _sel_wf(rep, a, k=KSEL_WF):
    c = rep.iloc[1:a].corrwith(y_wf.iloc[1:a]).abs().sort_values(ascending=False)
    return c.head(min(k, rep.shape[1])).index.tolist()
def _hi_wf(a): return float(y_wf.iloc[1:a].max()) * 1.5

def make_fitpred(rep, kind, ar=('y_lag1','y_lag2'), k=KSEL_WF, extra=None):
    def fp(a, b):
        cols = _sel_wf(rep, a, k); parts = []
        if 'y_lag1' in ar: parts.append(ylag1_wf.rename('y_lag1'))
        if 'y_lag2' in ar: parts.append(ylag2_wf.rename('y_lag2'))
        parts.append(rep[cols])
        if extra is not None: parts.append(extra)
        X = pd.concat(parts, axis=1).bfill(); s0 = 2 if len(ar) == 2 else 1
        Xtr, ytr, Xte = X.iloc[s0:a], y_wf.iloc[s0:a], X.iloc[a:b]
        if kind in ('ridge','enet','pls','mlr'):
            sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
            if kind == 'ridge': m = RidgeCV(alphas=[.1,1,10,100]).fit(Ztr, ytr); p = m.predict(Zte)
            elif kind == 'enet': m = ElasticNetCV(l1_ratio=[.1,.5,.9], cv=4, max_iter=5000, random_state=0).fit(Ztr, ytr); p = m.predict(Zte)
            elif kind == 'mlr':  m = LinearRegression().fit(Ztr, ytr); p = m.predict(Zte)
            else: m = PLSRegression(n_components=min(8, Xtr.shape[1])).fit(Ztr, ytr); p = m.predict(Zte).ravel()
        elif kind == 'rf':
            m = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(Xtr, ytr); p = m.predict(Xte)
        elif kind == 'gb':
            m = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, random_state=0).fit(Xtr, ytr); p = m.predict(Xte)
        else:
            m = xgb.XGBRegressor(max_depth=3, n_estimators=300, learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
                                 reg_lambda=3.0, reg_alpha=0.5, min_child_weight=10, random_state=0, n_jobs=-1).fit(Xtr, ytr); p = m.predict(Xte)
        return np.clip(p, 0.0, _hi_wf(a))
    return fp

def fit_ensemble(rep, extra=None):
    def fp(a, b): return 0.5*make_fitpred(rep,'enet',extra=extra)(a,b) + 0.5*make_fitpred(rep,'xgb',extra=extra)(a,b)
    return fp
def fit_persist(a, b): return np.clip(ylag1_wf.iloc[a:b].bfill().values, 0.0, _hi_wf(a))
def fit_media(a, b):   return np.full(b-a, float(y_wf.iloc[1:a].mean()))
def fit_ar1(a, b):
    m = RidgeCV(alphas=[.1,1,10]).fit(ylag1_wf.iloc[1:a].values.reshape(-1,1), y_wf.iloc[1:a].values)
    return np.clip(m.predict(ylag1_wf.iloc[a:b].values.reshape(-1,1)), 0.0, _hi_wf(a))
def make_arima(exog_rep=None, k_exog=8, order=(2,0,1)):
    def fp(a, b):
        ytr = y_wf.iloc[:a]
        if exog_rep is None:
            fit = SARIMAX(ytr, order=order, trend='c', enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            flt = SARIMAX(y_wf, order=order, trend='c', enforce_stationarity=False, enforce_invertibility=False).filter(fit.params)
            p = flt.get_prediction(start=a, end=b-1, dynamic=False).predicted_mean.values
        else:
            cols = _sel_wf(exog_rep, a, k_exog); ex = exog_rep[cols]
            mu, sd = ex.iloc[:a].mean(), ex.iloc[:a].std().replace(0,1.0); exz = (ex-mu)/sd
            fit = SARIMAX(ytr, exog=exz.iloc[:a], order=order, trend='c', enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=300)
            flt = SARIMAX(y_wf, exog=exz, order=order, trend='c', enforce_stationarity=False, enforce_invertibility=False).filter(fit.params)
            p = flt.get_prediction(start=a, end=b-1, exog=exz.iloc[a:b], dynamic=False).predicted_mean.values
        return np.clip(p, 0.0, _hi_wf(a))
    return fp

print(f'Toolkit pronto. Alvo: {N_wf} laudos | {N_FOLDS_WF} janelas | teste de {TESTE_INI_WF} a {N_wf}.')


# === Analise de residuos (walk-forward out-of-sample) ===
from scipy import stats as _stats
try:
    from statsmodels.graphics.tsaplots import plot_acf as _plot_acf
    from statsmodels.stats.diagnostic import acorr_ljungbox as _ljungbox
    _HAS_SM_RES = True
except Exception:
    _HAS_SM_RES = False

def analisar_residuos(nome, sufixo='', tol=0.3):
    """Diagnostico dos residuos (real - previsto) do walk-forward de um modelo.
    Metricas (vies, desvio, assimetria, curtose, ACF(1), Ljung-Box) + 6 graficos:
    residuo vs previsto, distribuicao, Q-Q, residuo vs tempo, ACF e real vs previsto."""
    if nome not in PREDICOES_WF:
        print(f'[residuos] sem predicoes para "{nome}" (rode o walk-forward do modelo antes)'); return None
    inst_oos, y_oos, p_oos = PREDICOES_WF[nome]
    y_oos = np.asarray(y_oos, float); p_oos = np.asarray(p_oos, float)
    e = y_oos - p_oos
    n = len(e); vies = float(e.mean()); dp = float(e.std(ddof=1)) if n > 1 else 0.0
    rmse = float(np.sqrt(np.mean(e**2))); mae = float(np.mean(np.abs(e)))
    skew = float(_stats.skew(e)); kurt = float(_stats.kurtosis(e))
    acf1 = float(np.corrcoef(e[:-1], e[1:])[0, 1]) if n > 2 else float('nan')
    dentro = float(np.mean(np.abs(e) <= tol) * 100)
    lb_p = float('nan')
    if _HAS_SM_RES and n > 12:
        try:
            lb_p = float(_ljungbox(e, lags=[10], return_df=True)['lb_pvalue'].iloc[0])
        except Exception:
            pass
    print(f"=== Residuos: {nome}{sufixo} ===")
    print(f"n={n} | vies(media)={vies:+.4f} | desvio={dp:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")
    print(f"assimetria={skew:+.3f} | curtose={kurt:+.3f} | ACF(1)={acf1:+.3f} | "
          f"Ljung-Box p(10)={lb_p:.4f} | dentro de +-{tol}: {dentro:.1f}%")

    fig, ax = plt.subplots(2, 3, figsize=(18, 9))
    ax[0, 0].scatter(p_oos, e, s=18, alpha=0.6, color='steelblue')
    ax[0, 0].axhline(0, color='red', lw=1, ls='--')
    ax[0, 0].set_title('Residuo vs Previsto'); ax[0, 0].set_xlabel('Previsto')
    ax[0, 0].set_ylabel('Residuo'); ax[0, 0].grid(True, ls='--', alpha=.5)

    ax[0, 1].hist(e, bins=30, density=True, color='steelblue', alpha=.7, edgecolor='white')
    _xs = np.linspace(float(e.min()), float(e.max()), 200)
    if dp > 0:
        ax[0, 1].plot(_xs, _stats.norm.pdf(_xs, vies, dp), 'r-', lw=1.5, label='Normal')
    ax[0, 1].axvline(0, color='k', lw=1, ls=':'); ax[0, 1].set_title('Distribuicao dos residuos')
    ax[0, 1].set_xlabel('Residuo'); ax[0, 1].legend(); ax[0, 1].grid(True, ls='--', alpha=.5)

    _stats.probplot(e, dist='norm', plot=ax[0, 2])
    ax[0, 2].set_title('Q-Q plot (normalidade)'); ax[0, 2].grid(True, ls='--', alpha=.5)

    xx, ee = _quebrar_paradas(inst_oos, e)
    ax[1, 0].plot(xx, ee, marker='o', ms=3, lw=.8, color='steelblue')
    ax[1, 0].axhline(0, color='red', lw=1, ls='--')
    _sombrear_paradas(ax[1, 0], pd.Timestamp(inst_oos[0]), pd.Timestamp(inst_oos[-1]))
    ax[1, 0].set_title('Residuo vs Tempo'); ax[1, 0].set_xlabel('Tempo')
    ax[1, 0].set_ylabel('Residuo'); ax[1, 0].grid(True, ls='--', alpha=.5)

    if _HAS_SM_RES and n > 12:
        _plot_acf(e, ax=ax[1, 1], lags=min(30, n // 2 - 1), zero=False)
        ax[1, 1].set_title('ACF dos residuos')
    else:
        ax[1, 1].text(.5, .5, 'ACF indisponivel', ha='center'); ax[1, 1].set_axis_off()

    _lim = [min(float(y_oos.min()), float(p_oos.min())), max(float(y_oos.max()), float(p_oos.max()))]
    ax[1, 2].scatter(y_oos, p_oos, s=18, alpha=.6, color='darkorange')
    ax[1, 2].plot(_lim, _lim, 'k--', lw=1)
    ax[1, 2].set_title('Real vs Previsto'); ax[1, 2].set_xlabel('Real')
    ax[1, 2].set_ylabel('Previsto'); ax[1, 2].grid(True, ls='--', alpha=.5)

    fig.suptitle(f'Analise de residuos - {nome}{sufixo}', fontsize=13)
    plt.tight_layout(); plt.show()
    return {'Modelo': nome, 'vies': vies, 'desvio': dp, 'RMSE': rmse, 'MAE': mae,
            'assimetria': skew, 'curtose': kurt, 'ACF(1)': acf1,
            'LjungBox_p10': lb_p, f'dentro_+-{tol}(%)': dentro}

## 📘 Validação walk-forward

A **validação walk-forward** (janela expansiva) respeita a ordem temporal: o modelo é treinado
apenas com o **passado** e avaliado no **futuro imediato**, nunca o contrário. Isso evita
*look-ahead* (vazamento de informação do futuro) — que uma validação cruzada embaralhada
produziria numa série temporal, superestimando o desempenho.

**Como funciona aqui:** a série de laudos é dividida em **5 janelas expansivas** a partir de
50% dos dados (`FOLDS_WF`). Em cada janela treina-se com tudo até o início do teste e prevê-se
o bloco seguinte, avançando pela série. Todas as previsões são *out-of-sample*.

**Métricas reportadas:**
- **R² médio** — média do R² das 5 janelas. Robusto a diferenças de regime entre janelas; é a
  **métrica de decisão** do projeto.
- **R² global** — R² sobre todas as previsões concatenadas; mais sensível a uma única janela ruim.
- **RMSE / MAE (médios)** — erro em unidades de CaO Livre (%). RMSE penaliza mais os erros
  grandes; MAE é o erro absoluto típico.
- R² ≈ 0 significa "não melhor que prever a média"; R² < 0 é **pior** que a média.

**Baselines de referência:** persistência (repetir o último laudo), média do treino e AR(1).
Um modelo só agrega valor se **superar a persistência** — que nesse base já é grande
(R² ≈ 0,58), pois o CaO Livre é bastante autocorrelacionado entre laudos consecutivos.


## 📊 Análise de resíduos

O **resíduo** é o erro do modelo: `err = real − previsto`, calculado sobre as previsões
*out-of-sample* do walk-forward. Um bom modelo tem resíduos **pequenos, centrados em zero,
sem padrão e sem autocorrelação temporal**.

**Métricas (o que cada uma indica):**
- **Viés (média dos resíduos)** — deve ser ≈ 0. Diferente de zero = erro sistemático (o modelo
  super ou subestima em média).
- **Desvio (dp)** — dispersão dos resíduos; quanto menor, mais consistente o modelo.
- **RMSE / MAE** — magnitude típica do erro (% de CaO). Se RMSE ≫ MAE, há erros grandes
  ocasionais (outliers).
- **Assimetria (skew)** — simetria da distribuição. ≈ 0 é simétrico; positivo → cauda de erros
  grandes para cima.
- **Curtose** — peso das caudas. ≈ 0 (excesso) ≈ normal; alta → outliers frequentes.
- **ACF(1)** — autocorrelação dos resíduos no lag 1. Deve ser ≈ 0; se alta, o modelo deixou
  **estrutura temporal** por explicar (o erro de um laudo "puxa" o do próximo).
- **Ljung-Box p(10)** — teste de autocorrelação nos 10 primeiros lags. **p alto (> 0,05) é BOM**
  (resíduos ≈ ruído branco); p baixo indica dinâmica residual não capturada.
- **% dentro de ±0,3** — fração dos laudos previstos dentro de ±0,3% de CaO (tolerância de
  referência). Leitura operacional direta.

**Os 6 gráficos:**
1. **Resíduo × Previsto** — nuvem sem padrão em torno de 0. Um "funil" (variância cresce com o
   previsto) indica heterocedasticidade.
2. **Distribuição + curva normal** — idealmente um sino centrado em 0.
3. **Q-Q plot** — pontos sobre a diagonal → resíduos normais; desvios nas pontas → caudas pesadas.
4. **Resíduo × Tempo** — procurar tendências/trechos ruins; áreas cinza = forno parado.
5. **ACF dos resíduos** — barras dentro da banda de confiança → sem autocorrelação (bom).
6. **Real × Previsto** — pontos sobre a diagonal → previsões acuradas.

**Pontos relevantes observados:** os modelos **ARIMA/ARIMAX** tendem a deixar resíduos
praticamente **brancos** (Ljung-Box p alto), enquanto os modelos de **árvore (RF/XGB/GB)**
deixam **forte autocorrelação residual** (p muito baixo) — sinal de que ainda há dinâmica
temporal a capturar.

# Experimento 1 - Features por lags de correlação

Cada variável de processo entra com o lag de maior informação mútua encontrada, somada à última medição de laboratório disponível.

## Construção do dataset com lags

In [ ]:
print("Construindo matriz de features alinhadas no tempo com lags ótimos de informação mútua...")

# Cria um novo DataFrame vazio mantendo o index original (minuto a minuto)
X_features = pd.DataFrame(index=df_dataset.index)

# 1. --- VARIÁVEIS DE PROCESSO ---
# Itera sobre a tabela de maiores correlações para aplicar o respectivo lag
for _, row in df_tabela_mi.iterrows():
    var_nome = row['Variável']
    lag_otimo = int(row['Lag (min)'])
    col_name = f"{var_nome}_lag{lag_otimo}"
    X_features[col_name] = df_dataset[var_nome].shift(lag_otimo)

# 2. --- ÚLTIMA MEDIÇÃO DO CaO livre ---
# Propaga a última medição válida (ffill) e desloca 1 minuto para o futuro (shift)
X_features['AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS_lag1'] = df_dataset[target_limpo].ffill().shift(1)

# 4. Adiciona a variável target limpa ao novo dataset
dataset_modelagem = X_features.join(df_dataset[target_limpo])

print(f"Dataset construído com {X_features.shape[1]} features.")
dataset_modelagem.head()

## Split temporal treino/teste

In [ ]:
# Isolamento das medições de laboratório e Train/Test Split

# Filtra APENAS os instantes onde temos a resposta do laboratório
df_modelos = dataset_modelagem.dropna(subset=[target_limpo]).copy()

# Remove as primeiras linhas que ficaram com NaN devido ao .shift() dos lags
df_modelos = df_modelos.dropna()

print(f"Total de amostras válidas para treinamento e validação: {len(df_modelos)}")

# Divisão Temporal — usa a MESMA data de corte da seleção de lags (sem vazamento)
train_set = df_modelos[df_modelos.index <= train_cutoff_date]
test_set = df_modelos[df_modelos.index > train_cutoff_date]

X_train = train_set.drop(columns=[target_limpo])
y_train = train_set[target_limpo]

X_test = test_set.drop(columns=[target_limpo])
y_test = test_set[target_limpo]

print(f"Amostras de Treino: {len(X_train)} | Amostras de Teste: {len(X_test)}")

# Plot da divisão para confirmação visual
plt.figure(figsize=(15, 4))
plt.plot(y_train.index, y_train, label='Treino', color='blue', marker='o', linestyle='', markersize=4)
plt.plot(y_test.index, y_test, label='Teste', color='orange', marker='o', linestyle='', markersize=4)
plt.title('Divisão Temporal: Treino vs Teste')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Padronização das features

O `StandardScaler` é ajustado apenas no treino para evitar vazamento de dados.

In [ ]:
# Padronização das features
from sklearn.preprocessing import StandardScaler

# O scaler é ajustado APENAS no treino para evitar vazamento de dados
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)

# O teste é transformado usando as regras aprendidas no treino
X_test_scaled = scaler_X.transform(X_test)

## Validacao walk-forward por modelo (Exp1)

Alem do split unico, cada modelo abaixo recebe a validacao walk-forward (metricas + grafico Real x Previsto). A representacao de lags e construida uma vez:

In [ ]:
col_var = 'Variável' if 'Variável' in df_tabela_mi.columns else df_tabela_mi.columns[1]
rep_lags = pd.concat({f'{r[col_var]}_lag{int(r["Lag (min)"])}': df_dataset[r[col_var]].shift(int(r['Lag (min)']))
                      for _, r in df_tabela_mi.iterrows()}, axis=1).loc[inst_wf].reset_index(drop=True).ffill().bfill().fillna(0)
print('rep_lags (lags por Informacao Mutua):', rep_lags.shape)

## Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

# Treina a regressão Ridge e gera previsões em treino e teste
modelo_ridge = Ridge(alpha=70.0)
modelo_ridge.fit(X_train_scaled, y_train)

preds_treino_ridge = modelo_ridge.predict(X_train_scaled)
preds_teste_ridge = modelo_ridge.predict(X_test_scaled)

avaliar_modelo("Ridge (lags)", y_train, preds_treino_ridge, y_test, preds_teste_ridge)

In [ ]:
plotar_resultados(y_train, preds_treino_ridge, y_test, preds_teste_ridge, "Ridge (lags)")

In [ ]:
# Validacao walk-forward - Lags + Ridge
avaliar_wf('Exp1 - Lags (MI)', 'Lags + Ridge', make_fitpred(rep_lags, 'ridge'))
plotar_real_previsto('Lags + Ridge')

### Análise de resíduos

In [ ]:
analisar_residuos('Lags + Ridge')

## PLS (Partial Least Squares)

In [ ]:
from sklearn.cross_decomposition import PLSRegression

modelo_pls = PLSRegression(n_components=12)
modelo_pls.fit(X_train_scaled, y_train)

# flatten() porque o PLS retorna um array 2D
preds_treino_pls = modelo_pls.predict(X_train_scaled).flatten()
preds_teste_pls = modelo_pls.predict(X_test_scaled).flatten()

avaliar_modelo("PLS (lags)", y_train, preds_treino_pls, y_test, preds_teste_pls)

In [ ]:
plotar_resultados(y_train, preds_treino_pls, y_test, preds_teste_pls, "PLS (lags)")

In [ ]:
# Validacao walk-forward - Lags + PLS
avaliar_wf('Exp1 - Lags (MI)', 'Lags + PLS', make_fitpred(rep_lags, 'pls'))
plotar_real_previsto('Lags + PLS')

### Analise de residuos

In [ ]:
analisar_residuos('Lags + PLS')

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# max_depth e min_samples_leaf limitam a memorização de ruído
modelo_rf = RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=4, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)

preds_treino_rf = modelo_rf.predict(X_train_scaled)
preds_teste_rf = modelo_rf.predict(X_test_scaled)

avaliar_modelo("Random Forest (lags)", y_train, preds_treino_rf, y_test, preds_teste_rf)

In [ ]:
plotar_resultados(y_train, preds_treino_rf, y_test, preds_teste_rf, "Random Forest (lags)")

In [ ]:
# Validacao walk-forward - Lags + Random Forest
avaliar_wf('Exp1 - Lags (MI)', 'Lags + Random Forest', make_fitpred(rep_lags, 'rf'))
plotar_real_previsto('Lags + Random Forest')

### Analise de residuos

In [ ]:
analisar_residuos('Lags + Random Forest')

### Importância das variáveis

In [ ]:
# 15 variáveis mais determinantes para o Random Forest
df_importancia = tabela_importancias(modelo_rf, X_train.columns, top=15)
print("As 15 variáveis mais determinantes para o Soft Sensor:")
display(df_importancia.style.background_gradient(cmap='Greens', subset=['Importancia (%)']))

## Gradient Boosting

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# l2_regularization penaliza variações bruscas, combatendo o overfitting
modelo_hgb = HistGradientBoostingRegressor(max_iter=150, max_depth=8, l2_regularization=0.2, random_state=42)
modelo_hgb.fit(X_train_scaled, y_train)

preds_treino_hgb = modelo_hgb.predict(X_train_scaled)
preds_teste_hgb = modelo_hgb.predict(X_test_scaled)

avaliar_modelo("Gradient Boosting (lags)", y_train, preds_treino_hgb, y_test, preds_teste_hgb)

In [ ]:
plotar_resultados(y_train, preds_treino_hgb, y_test, preds_teste_hgb, "Gradient Boosting (lags)")

In [ ]:
# Validacao walk-forward - Lags + Gradient Boosting
avaliar_wf('Exp1 - Lags (MI)', 'Lags + Gradient Boosting', make_fitpred(rep_lags, 'gb'))
plotar_real_previsto('Lags + Gradient Boosting')

### Analise de residuos

In [ ]:
analisar_residuos('Lags + Gradient Boosting')

## XGBoost

In [ ]:
import xgboost as xgb

# Parâmetros restritivos (profundidade baixa + L2 forte) para evitar memorizar ruído
modelo_xgb = xgb.XGBRegressor(
    n_estimators=150, max_depth=4, learning_rate=0.05, reg_lambda=5.0, random_state=42,
)
modelo_xgb.fit(X_train_scaled, y_train)

preds_treino_xgb = modelo_xgb.predict(X_train_scaled)
preds_teste_xgb = modelo_xgb.predict(X_test_scaled)

avaliar_modelo("XGBoost (lags)", y_train, preds_treino_xgb, y_test, preds_teste_xgb)

In [ ]:
plotar_resultados(y_train, preds_treino_xgb, y_test, preds_teste_xgb, "XGBoost (lags)")

In [ ]:
# Validacao walk-forward - Lags + XGBoost
avaliar_wf('Exp1 - Lags (MI)', 'Lags + XGBoost', make_fitpred(rep_lags, 'xgb'))
plotar_real_previsto('Lags + XGBoost')

### Analise de residuos

In [ ]:
analisar_residuos('Lags + XGBoost')

## Regressão Linear Múltipla (MLR)

In [ ]:
# Treinamento da Regressão Linear Múltipla (MLR)
from sklearn.linear_model import LinearRegression

print("Treinando modelo de Regressão Linear Múltipla...")

modelo_mlr = LinearRegression()
modelo_mlr.fit(X_train_scaled, y_train)

# Previsões
preds_treino_mlr = modelo_mlr.predict(X_train_scaled)
preds_teste_mlr = modelo_mlr.predict(X_test_scaled)

# Avaliação padronizada
avaliar_modelo("MLR (Lags)", y_train, preds_treino_mlr, y_test, preds_teste_mlr)

In [ ]:
# Gráficos de Real vs Previsto
plotar_resultados(y_train, preds_treino_mlr, y_test, preds_teste_mlr, "Regressão Linear Múltipla (Lags)")

In [ ]:
# Validacao walk-forward - Lags + MLR
avaliar_wf('Exp1 - Lags (MI)', 'Lags + MLR', make_fitpred(rep_lags, 'mlr'))
plotar_real_previsto('Lags + MLR')

### Analise de residuos

In [ ]:
analisar_residuos('Lags + MLR')

## Verificação de Concept Drift

Compara as distribuições do target entre treino e teste para detectar mudança de regime.

In [ ]:
# Análise de mudança de regime (concept drift) entre treino e teste
import seaborn as sns

# 1. Estatísticas descritivas comparativas
print("--- Comparativo Estatístico do Target (Laboratório) ---")
df_comparativo = pd.DataFrame({
    'Base de Treino': y_train.describe(),
    'Base de Teste': y_test.describe()
})
display(df_comparativo.style.format("{:.4f}"))

# 2. Visualização das distribuições
plt.figure(figsize=(12, 5))
sns.kdeplot(y_train, label='Distribuição Treino', fill=True, color='blue', alpha=0.4)
sns.kdeplot(y_test, label='Distribuição Teste', fill=True, color='orange', alpha=0.4)

# Linhas de média
plt.axvline(y_train.mean(), color='darkblue', linestyle='--', label=f'Média Treino: {y_train.mean():.2f}')
plt.axvline(y_test.mean(), color='darkorange', linestyle='--', label=f'Média Teste: {y_test.mean():.2f}')

plt.title('Diagnóstico de Concept Drift no Target')
plt.xlabel('Valores do Target')
plt.ylabel('Densidade')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Avaliacao walk-forward - Experimento 1 (Lags por Informacao Mutua)

Metricas padronizadas, validacao walk-forward e grafico Real x Previsto do **melhor R2** desta secao.

In [ ]:
# Consolidacao walk-forward - Experimento 1 (Lags por Informacao Mutua) (compara os modelos da secao, sem recalcular)
df_wf_exp1 = tabela_wf('Exp1 - Lags (MI)')
print('Comparativo walk-forward - Experimento 1 (Lags por Informacao Mutua):')
display(df_wf_exp1.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df_wf_exp1.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
_melhor = df_wf_exp1.iloc[0]['Modelo']
print('>>> Melhor da secao (R2 médio):', _melhor)
plotar_real_previsto(_melhor, sufixo='  [melhor da secao]')

# Experimento 2 - Features por janelas estatísticas

Em vez de lags pontuais, resume cada variável por estatísticas móveis (média, desvio, mínimo, máximo) em janelas de 180 minutos.

## Feature Engineering (janelas de 180 min)

In [ ]:
# Extração de Estatísticas em Múltiplas Janelas Temporais
import pandas as pd

print("Calculando features para janelas...")

janelas = [180]
features_processo = [col for col in df_dataset.columns if col not in [target_col, target_limpo]]

lista_dfs_janelas = []

# Gera o rolling para cada janela e anexa à lista
for j in janelas:
    df_temp = df_dataset[features_processo].rolling(window=j).agg(['mean', 'std', 'min', 'max'])
    # Adiciona o indicador da janela no nome da coluna (ex: sensorX_mean_180m)
    df_temp.columns = [f"{col[0]}_{col[1]}_{j}m" for col in df_temp.columns]
    lista_dfs_janelas.append(df_temp)

# Concatena todas as 4 bases horizontalmente
df_multi_janelas = pd.concat(lista_dfs_janelas, axis=1)

# Propaga a última medição válida (ffill) e desloca 1 minuto para o futuro (shift)
df_multi_janelas['AR.F1.CLINQUER_CaO_LIVRE_SRESF_LIMS_lag1'] = df_dataset[target_limpo].ffill().shift(1)

# Insere a variável target limpa e filtra APENAS as linhas de laudo
df_multi_janelas[target_limpo] = df_dataset[target_limpo]
df_modelos_multi = df_multi_janelas.dropna(subset=[target_limpo]).copy()

# Remove as amostras iniciais baseadas na MAIOR janela (180 min)
df_modelos_multi = df_modelos_multi.dropna()

print(f"Base multi-janelas criada: {df_modelos_multi.shape[0]} amostras com {df_modelos_multi.shape[1]-1} features.")

## Split temporal treino/teste

In [ ]:
# Divisão Cronológica (Train/Test Split)
split_index = int(len(df_modelos_multi) * 0.8)

train_set = df_modelos_multi.iloc[:split_index]
test_set = df_modelos_multi.iloc[split_index:]

X_train_multi = train_set.drop(columns=[target_limpo])
y_train_win = train_set[target_limpo]

X_test_multi = test_set.drop(columns=[target_limpo])
y_test_win = test_set[target_limpo]

print(f"Divisão -> Treino: {X_train_multi.shape[0]} amostras | Teste: {X_test_multi.shape[0]} amostras")

## Feature Selection

In [ ]:
# Feature Selection com Random Forest
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor

print("Executando Feature Selection...")

# Instancia um Random Forest como "avaliador" de importância
avaliador_rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)

# O seletor mantém as features de importância maior que a mediana (threshold)
seletor = SelectFromModel(avaliador_rf, threshold='median')
seletor.fit(X_train_multi, y_train_win)

# Captura os nomes das features aprovadas
features_aprovadas = X_train_multi.columns[seletor.get_support()]

print(f"Total de features ORIGINAIS: {X_train_multi.shape[1]}")
print(f"Total de features MANTIDAS: {len(features_aprovadas)}")

# Filtra os DataFrames mantendo apenas as features que passaram no corte
X_train_sel = X_train_multi[features_aprovadas]
X_test_sel = X_test_multi[features_aprovadas]

# Exibe as Top 15 para entendermos quais tempos de janela o processo mais "escuta"
df_importancias = pd.DataFrame({
    'Feature': X_train_multi.columns, 
    'Importância': seletor.estimator_.feature_importances_
}).sort_values(by='Importância', ascending=False)

display(df_importancias.head(15))

## Padronização das features

In [ ]:
# Célula 4: Padronização da Base Selecionada (Scaling)
from sklearn.preprocessing import StandardScaler

scaler_sel = StandardScaler()

X_train_scaled = scaler_sel.fit_transform(X_train_sel)
X_test_scaled = scaler_sel.transform(X_test_sel)

# Reconstrução para DataFrame mantendo index e colunas
X_train_win_scaled = pd.DataFrame(X_train_scaled, index=X_train_sel.index, columns=X_train_sel.columns)
X_test_win_scaled = pd.DataFrame(X_test_scaled, index=X_test_sel.index, columns=X_test_sel.columns)

print("Matrizes prontas para o treinamento dos modelos!")

## Validacao walk-forward por modelo (Exp2)

Mesmo padrao do Exp1: cada modelo recebe a walk-forward. A representacao de janelas (e as exogenas do ARIMAX) e construida uma vez:

In [ ]:
_proc_cols = [c for c in df_dataset.columns if c not in [target_col, target_limpo]]
_roll = df_dataset[_proc_cols].rolling(180, min_periods=30)
rep_jan = pd.concat([_roll.mean().add_suffix('_mean180'), _roll.std().add_suffix('_std180'),
                     _roll.min().add_suffix('_min180'),  _roll.max().add_suffix('_max180')], axis=1
                   ).loc[inst_wf].reset_index(drop=True).ffill().bfill().fillna(0)
rep_mean180 = rep_jan[[c for c in rep_jan.columns if c.endswith('_mean180')]]   # exogenas p/ ARIMAX (Exp4)
print('rep_jan (janelas 180 min):', rep_jan.shape)

## Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

modelo_ridge = Ridge(alpha=50)
modelo_ridge.fit(X_train_win_scaled, y_train_win)

preds_treino_ridge = modelo_ridge.predict(X_train_win_scaled)
preds_teste_ridge = modelo_ridge.predict(X_test_win_scaled)

avaliar_modelo("Ridge (janelas)", y_train_win, preds_treino_ridge, y_test_win, preds_teste_ridge)

In [ ]:
plotar_resultados(y_train_win, preds_treino_ridge, y_test_win, preds_teste_ridge, "Ridge (janelas)")

In [ ]:
# Validacao walk-forward - Janelas + Ridge
avaliar_wf('Exp2 - Janelas 180', 'Janelas + Ridge', make_fitpred(rep_jan, 'ridge'))
plotar_real_previsto('Janelas + Ridge')

### Analise de residuos

In [ ]:
analisar_residuos('Janelas + Ridge')

## PLS (Partial Least Squares)

In [ ]:
from sklearn.cross_decomposition import PLSRegression

modelo_pls = PLSRegression(n_components=5)
modelo_pls.fit(X_train_win_scaled, y_train_win)

preds_treino_pls = modelo_pls.predict(X_train_win_scaled).flatten()
preds_teste_pls = modelo_pls.predict(X_test_win_scaled).flatten()

avaliar_modelo("PLS (janelas)", y_train_win, preds_treino_pls, y_test_win, preds_teste_pls)

In [ ]:
plotar_resultados(y_train_win, preds_treino_pls, y_test_win, preds_teste_pls, "PLS (janelas)")

In [ ]:
# Validacao walk-forward - Janelas + PLS
avaliar_wf('Exp2 - Janelas 180', 'Janelas + PLS', make_fitpred(rep_jan, 'pls'))
plotar_real_previsto('Janelas + PLS')

### Analise de residuos

In [ ]:
analisar_residuos('Janelas + PLS')

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modelo_rf = RandomForestRegressor(n_estimators=150, max_depth=5, min_samples_leaf=10,
                                  max_features='sqrt', random_state=42)
modelo_rf.fit(X_train_win_scaled, y_train_win)

preds_treino_rf = modelo_rf.predict(X_train_win_scaled)
preds_teste_rf = modelo_rf.predict(X_test_win_scaled)

avaliar_modelo("Random Forest (janelas)", y_train_win, preds_treino_rf, y_test_win, preds_teste_rf)

In [ ]:
plotar_resultados(y_train_win, preds_treino_rf, y_test_win, preds_teste_rf, "Random Forest (janelas)")

In [ ]:
# Validacao walk-forward - Janelas + Random Forest
avaliar_wf('Exp2 - Janelas 180', 'Janelas + Random Forest', make_fitpred(rep_jan, 'rf'))
plotar_real_previsto('Janelas + Random Forest')

### Analise de residuos

In [ ]:
analisar_residuos('Janelas + Random Forest')

## XGBoost

In [ ]:
import xgboost as xgb

modelo_xgb = xgb.XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05,
                              reg_lambda=10.0, random_state=42)
modelo_xgb.fit(X_train_win_scaled, y_train_win)

preds_treino_xgb = modelo_xgb.predict(X_train_win_scaled)
preds_teste_xgb = modelo_xgb.predict(X_test_win_scaled)

avaliar_modelo("XGBoost (janelas)", y_train_win, preds_treino_xgb, y_test_win, preds_teste_xgb)

In [ ]:
plotar_resultados(y_train_win, preds_treino_xgb, y_test_win, preds_teste_xgb, "XGBoost (janelas)")

In [ ]:
# Validacao walk-forward - Janelas + XGBoost
avaliar_wf('Exp2 - Janelas 180', 'Janelas + XGBoost', make_fitpred(rep_jan, 'xgb'))
plotar_real_previsto('Janelas + XGBoost')

### Analise de residuos

In [ ]:
analisar_residuos('Janelas + XGBoost')

## Avaliacao walk-forward - Experimento 2 (Janelas estatisticas)

Metricas padronizadas, walk-forward e grafico Real x Previsto do **melhor R2** desta secao.

In [ ]:
# Consolidacao walk-forward - Experimento 2 (Janelas estatisticas) (compara os modelos da secao, sem recalcular)
df_wf_exp2 = tabela_wf('Exp2 - Janelas 180')
print('Comparativo walk-forward - Experimento 2 (Janelas estatisticas):')
display(df_wf_exp2.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df_wf_exp2.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
_melhor = df_wf_exp2.iloc[0]['Modelo']
print('>>> Melhor da secao (R2 médio):', _melhor)
plotar_real_previsto(_melhor, sufixo='  [melhor da secao]')

# Experimento 3 - Deep Learning (LSTM sequencial)

Redes **LSTM** que leem a janela temporal de processo (sequência 3D) antes de cada laudo. Esta
seção foi **reescrita** para corrigir erros de arquitetura e de avaliação, e passa a ser avaliada
no **mesmo protocolo walk-forward** dos demais experimentos — de modo a julgar de forma justa se
há ganho ao evoluir para Deep Learning.

**O que estava errado na versão anterior (4 LSTMs em holdout):**
- Avaliação por **holdout 80/20 isolado**, incompatível com o walk-forward dos outros experimentos
  (não comparável e fora do comparativo final).
- **`InstanceNorm` aplicada a todos os canais**, inclusive o do último laudo (`y_lag1`, quase
  constante na janela): a normalização por janela dividia por um desvio ≈ 0 e **destruía o sinal
  mais preditivo** — principal causa do R² negativo.
- **Vazamento de borda** (`bfill` global preenchia buracos com o futuro) e ausência de
  escalonamento ajustado só no treino.
- Seções reaproveitando tensores umas das outras; sem seleção de modelo / early stopping.

**Correções aplicadas nesta versão:**
- **Walk-forward de 5 janelas** (`avaliar_wf`) → comparável e entra no comparativo final + análise
  de resíduos.
- **Sem vazamento:** `StandardScaler` (processo, AR e alvo) ajustado **apenas no treino de cada
  fold**; janelas usam só o passado (`ffill`, sem `bfill` global); **early stopping** por uma
  validação interna do treino (nunca o teste).
- **Backbone AR preservado:** `y_lag1`/`y_lag2` entram **concatenados após a LSTM**, sem passar
  pela normalização de sequência.
- **Treino embaralhado**, gradient clipping e semente fixa.
- Janela de 180 min **subamostrada a cada 5 min** (36 passos) para uma sequência treinável.


## Preparação: sequências 3D + backbone AR (leakage-free)

Para cada laudo, monta-se a janela das features de processo dos últimos `JANELA_MIN` minutos
(1 passo a cada `PASSO_MIN` min) e separam-se os termos autorregressivos `y_lag1`/`y_lag2`. O
escalonamento é feito **dentro de cada fold** (só no treino), então aqui montamos apenas os
tensores brutos e definimos a rede e a função de treino/predição usada pelo walk-forward.


In [ ]:
# --- Sequencias 3D + AR + arquitetura LSTM (usadas pelo walk-forward) ---
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
torch.manual_seed(0); np.random.seed(0)

JANELA_MIN, PASSO_MIN = 180, 5          # janela de contexto e subamostragem
SEQ_LEN = JANELA_MIN // PASSO_MIN        # 36 passos

# features de processo (minuto a minuto, ffill = so passado; NaN inicial -> 0)
_feat_proc = [c for c in df_dataset.columns if c not in [target_col, target_limpo]]
_mat = df_dataset[_feat_proc].ffill().values.astype(np.float32)
_pos = df_dataset.index.get_indexer(inst_wf)     # posicao de cada laudo no indice de minutos
F = len(_feat_proc)
_off = np.arange(-(SEQ_LEN - 1) * PASSO_MIN, 1, PASSO_MIN)   # [-175,...,-5,0]

X_seq_wf = np.zeros((N_wf, SEQ_LEN, F), dtype=np.float32)
for i, _p in enumerate(_pos):
    X_seq_wf[i] = _mat[np.clip(_p + _off, 0, len(_mat) - 1)]  # pad no inicio da serie
X_seq_wf = np.nan_to_num(X_seq_wf, nan=0.0)

# backbone autorregressivo (ultimos laudos) alinhado aos instantes do walk-forward
AR_wf = np.nan_to_num(np.column_stack([ylag1_wf.values, ylag2_wf.values]).astype(np.float32))
print(f'Sequencias: X_seq_wf {X_seq_wf.shape} (janela {JANELA_MIN}min/{PASSO_MIN}min) | AR_wf {AR_wf.shape}')


class SoftSensorLSTM(nn.Module):
    """LSTM sobre as features de processo; y_lag1/y_lag2 entram concatenados APOS a LSTM."""
    def __init__(self, n_proc, n_ar, hidden=32, layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_proc, hidden, layers, batch_first=True,
                            dropout=(dropout if layers > 1 else 0.0))
        self.head = nn.Sequential(nn.Linear(hidden + n_ar, 32), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, xs, xa):
        out, _ = self.lstm(xs)          # [B, T, H]
        return self.head(torch.cat([out[:, -1, :], xa], dim=1))


def make_lstm_fitpred(hidden=32, layers=1, dropout=0.2, epocas=60, paciencia=8,
                      lr=1e-3, batch=64, seed=0):
    """Retorna fitpred(a,b) compativel com avaliar_wf: treina so com [.. :a] (com validacao
    interna p/ early stopping) e preve [a:b]. Escalonamento ajustado SO no treino de cada fold."""
    def fp(a, b):
        s0 = 2                                   # y_lag1/y_lag2 exigem 2 laudos previos
        n_tr = a - s0
        v0 = s0 + max(1, int(n_tr * 0.85))       # ultimos 15% do treino = validacao (temporal)
        tr, va, te = np.arange(s0, v0), np.arange(v0, a), np.arange(a, b)

        fsc = StandardScaler().fit(X_seq_wf[tr].reshape(-1, F))     # so no treino
        asc = StandardScaler().fit(AR_wf[tr])
        ysc = StandardScaler().fit(y_wf.values[tr].reshape(-1, 1))

        def prep(idx):
            xs = fsc.transform(X_seq_wf[idx].reshape(-1, F)).reshape(len(idx), SEQ_LEN, F)
            return (torch.tensor(xs, dtype=torch.float32),
                    torch.tensor(asc.transform(AR_wf[idx]), dtype=torch.float32))

        Xtr_s, Xtr_a = prep(tr); Xva_s, Xva_a = prep(va); Xte_s, Xte_a = prep(te)
        ytr = torch.tensor(ysc.transform(y_wf.values[tr].reshape(-1, 1)), dtype=torch.float32)
        yva = torch.tensor(ysc.transform(y_wf.values[va].reshape(-1, 1)), dtype=torch.float32)

        torch.manual_seed(seed)
        net = SoftSensorLSTM(F, AR_wf.shape[1], hidden, layers, dropout)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        lossf = nn.MSELoss()
        n = len(tr); melhor = (1e9, None); espera = 0
        for _ in range(epocas):
            net.train(); perm = torch.randperm(n)
            for k in range(0, n, batch):
                j = perm[k:k + batch]
                opt.zero_grad()
                loss = lossf(net(Xtr_s[j], Xtr_a[j]), ytr[j]); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            net.eval()
            with torch.no_grad():
                vl = lossf(net(Xva_s, Xva_a), yva).item() if len(va) else 0.0
            if vl < melhor[0] - 1e-5:
                melhor = (vl, {k: v.clone() for k, v in net.state_dict().items()}); espera = 0
            else:
                espera += 1
                if espera >= paciencia:
                    break
        if melhor[1] is not None:
            net.load_state_dict(melhor[1])       # restaura o melhor (early stopping)
        net.eval()
        with torch.no_grad():
            p = net(Xte_s, Xte_a).numpy().ravel()
        return np.clip(ysc.inverse_transform(p.reshape(-1, 1)).ravel(), 0.0, _hi_wf(a))
    return fp


## LSTM (walk-forward)

In [ ]:
# Avalia a LSTM no walk-forward de 5 janelas (retreino por fold, leakage-free)
avaliar_wf('Exp3 - LSTM', 'LSTM (seq+AR)', make_lstm_fitpred())
plotar_real_previsto('LSTM (seq+AR)')

### Analise de residuos

In [ ]:
analisar_residuos('LSTM (seq+AR)')

## Arquiteturas alternativas de DL (mesmo walk-forward)

Mantendo a LSTM base, testamos abordagens indicadas para regressão sequencial com variáveis
exógenas, todas no **mesmo protocolo walk-forward** e com o mesmo tratamento leakage-free:

- **TCN** (convolução causal dilatada) — capta padrões multi-escala no tempo, alternativa moderna à LSTM.
- **GRU** — recorrente mais enxuta (menos parâmetros → menos overfitting em dados escassos).
- **LSTM + atenção temporal** — pondera os 36 passos da janela em vez de usar só o último.
- **LSTM differencing** — prevê a *variação* Δ em relação ao último laudo (foco nas variações do alvo).
- **LSTM residual (base ARX)** — a rede aprende só o **resíduo** de um baseline linear ARX
  (AR + exógenas), i.e., o que o modelo autorregressivo não explica (*residual learning*).

Um framework único define os encoders e a função de treino/predição; os alvos `level`/`diff`/`residual`
apenas mudam o que a rede aprende (valor, variação ou resíduo do baseline).

In [ ]:
# --- Framework de DL: encoders (LSTM/GRU/Attn/TCN) + alvos (level/diff/residual) ---
import numpy as np, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

_proc_summary = X_seq_wf.mean(axis=1)   # media da janela por feature [N_wf, F] (exogenas p/ o baseline ARX)


class _Chomp(nn.Module):
    """Remove o padding a direita para manter a convolucao causal (nao ve o futuro)."""
    def __init__(self, s): super().__init__(); self.s = s
    def forward(self, x): return x[:, :, :-self.s].contiguous() if self.s > 0 else x


class _TCN(nn.Module):
    """Pilha de convolucoes causais dilatadas (dilatacao 1,2,4)."""
    def __init__(self, n_proc, hidden, dropout, levels=3, k=3):
        super().__init__()
        camadas, c = [], n_proc
        for i in range(levels):
            d = 2 ** i; pad = (k - 1) * d
            camadas += [nn.Conv1d(c, hidden, k, padding=pad, dilation=d), _Chomp(pad),
                        nn.ReLU(), nn.Dropout(dropout)]
            c = hidden
        self.net = nn.Sequential(*camadas)
    def forward(self, x):                 # x [B, T, F]
        return self.net(x.transpose(1, 2))[:, :, -1]   # [B, hidden] (ultimo passo)


class DLSoftSensor(nn.Module):
    """Encoder da sequencia de processo (por arch) + concatenacao do backbone AR APOS o encoder."""
    def __init__(self, arch, n_proc, n_ar, hidden=32, layers=1, dropout=0.2, seq_len=36):
        super().__init__(); self.arch = arch
        if arch in ('lstm', 'attn'):
            self.rnn = nn.LSTM(n_proc, hidden, layers, batch_first=True, dropout=(dropout if layers > 1 else 0.0))
        elif arch == 'gru':
            self.rnn = nn.GRU(n_proc, hidden, layers, batch_first=True, dropout=(dropout if layers > 1 else 0.0))
        elif arch == 'tcn':
            self.tcn = _TCN(n_proc, hidden, dropout)
        if arch == 'attn':
            self.attn = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
        self.head = nn.Sequential(nn.Linear(hidden + n_ar, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
    def _encode(self, xs):
        if self.arch == 'tcn':
            return self.tcn(xs)
        out, _ = self.rnn(xs)
        if self.arch == 'attn':
            w = torch.softmax(self.attn(out), dim=1)   # peso por passo temporal
            return torch.sum(w * out, dim=1)
        return out[:, -1, :]
    def forward(self, xs, xa):
        return self.head(torch.cat([self._encode(xs), xa], dim=1))


def _arx_baseline(s0, a, b, k=8, psum=None):
    """Baseline linear ARX (AR + top-k exogenas por corr no treino), ajustado SO no treino.
    Retorna previsoes para os indices [s0:b] (stand-in do ARIMAX p/ o residual learning)."""
    yv = y_wf.values; tr = np.arange(s0, a)
    PS = _proc_summary if psum is None else psum
    Xtr, ytr = PS[tr], yv[tr]
    cc = np.array([abs(np.corrcoef(Xtr[:, j], ytr)[0, 1]) if np.std(Xtr[:, j]) > 1e-9 else 0.0 for j in range(PS.shape[1])])
    sel = np.argsort(cc)[::-1][:k]
    design = lambda idx: np.column_stack([AR_wf[idx], PS[idx][:, sel]])
    sc = StandardScaler().fit(design(tr)); reg = Ridge(alpha=10.0).fit(sc.transform(design(tr)), ytr)
    out = np.zeros(N_wf); allidx = np.arange(s0, b)
    out[allidx] = reg.predict(sc.transform(design(allidx)))
    return out


_exog_df = pd.DataFrame(_proc_summary, columns=_feat_proc)   # exogenas (medias de janela) p/ o ARIMAX


def _arimax_baseline(s0, a, b, k_exog=8, order=(2, 0, 1)):
    """Baseline ARIMAX real (SARIMAX + top-k exogenas), ajustado SO no treino [:a]; previsoes
    1-passo p/ [s0:b] (teacher-forcing, como os demais baselines de AR). Fallback p/ ARX se falhar."""
    if not _HAS_SM:
        return _arx_baseline(s0, a, b, k_exog)
    try:
        out = np.zeros(N_wf)
        cols = _sel_wf(_exog_df, a, k_exog); ex = _exog_df[cols]
        mu, sd = ex.iloc[:a].mean(), ex.iloc[:a].std().replace(0, 1.0); exz = (ex - mu) / sd
        fit = SARIMAX(y_wf.iloc[:a], exog=exz.iloc[:a], order=order, trend='c',
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=300)
        flt = SARIMAX(y_wf, exog=exz, order=order, trend='c',
                      enforce_stationarity=False, enforce_invertibility=False).filter(fit.params)
        out[s0:b] = flt.get_prediction(start=s0, end=b - 1, dynamic=False).predicted_mean.values
        return out
    except Exception:
        return _arx_baseline(s0, a, b, k_exog)


def make_dl_fitpred(arch='lstm', alvo='level', hidden=32, layers=1, dropout=0.2,
                    epocas=60, paciencia=8, lr=1e-3, batch=64, seed=0, k_exog=8, seq=None, psum=None,
                    residual_base='arx', n_seeds=1):
    """fitpred(a,b) p/ avaliar_wf. alvo: 'level' (preve y), 'diff' (preve y-y_lag1),
    'residual' (preve y - baseline). residual_base: 'arx' (linear) ou 'arimax' (SARIMAX real).
    n_seeds>1 => ensemble (media de n redes com sementes distintas). Ajustes SO no treino."""
    def fp(a, b):
        _X = X_seq_wf if seq is None else seq
        _F = _X.shape[2]
        s0 = 2; n_tr = a - s0; v0 = s0 + max(1, int(n_tr * 0.85))
        tr, va, te = np.arange(s0, v0), np.arange(v0, a), np.arange(a, b)
        yv = y_wf.values
        if alvo == 'level':
            base = np.zeros(N_wf)
        elif alvo == 'diff':
            base = np.nan_to_num(ylag1_wf.values).astype(float)     # ultimo laudo
        else:
            base = (_arimax_baseline(s0, a, b, k_exog) if residual_base == 'arimax'
                    else _arx_baseline(s0, a, b, k_exog, psum))     # residual learning
        z = yv - base                                               # alvo que o DL aprende

        fsc = StandardScaler().fit(_X[tr].reshape(-1, _F))
        asc = StandardScaler().fit(AR_wf[tr]); zsc = StandardScaler().fit(z[tr].reshape(-1, 1))
        def prep(idx):
            xs = fsc.transform(_X[idx].reshape(-1, _F)).reshape(len(idx), SEQ_LEN, _F)
            return torch.tensor(xs, dtype=torch.float32), torch.tensor(asc.transform(AR_wf[idx]), dtype=torch.float32)
        Xtr_s, Xtr_a = prep(tr); Xva_s, Xva_a = prep(va); Xte_s, Xte_a = prep(te)
        ztr = torch.tensor(zsc.transform(z[tr].reshape(-1, 1)), dtype=torch.float32)
        zva = torch.tensor(zsc.transform(z[va].reshape(-1, 1)), dtype=torch.float32)

        zhats = []
        for _sd in range(n_seeds):                                  # ensemble de sementes
            torch.manual_seed(seed + _sd)
            net = DLSoftSensor(arch, _F, AR_wf.shape[1], hidden, layers, dropout, SEQ_LEN)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4); lossf = nn.MSELoss()
            n = len(tr); melhor = (1e9, None); espera = 0
            for _ in range(epocas):
                net.train(); perm = torch.randperm(n)
                for kk in range(0, n, batch):
                    j = perm[kk:kk + batch]; opt.zero_grad()
                    loss = lossf(net(Xtr_s[j], Xtr_a[j]), ztr[j]); loss.backward()
                    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
                net.eval()
                with torch.no_grad():
                    vl = lossf(net(Xva_s, Xva_a), zva).item() if len(va) else 0.0
                if vl < melhor[0] - 1e-5:
                    melhor = (vl, {k: v.clone() for k, v in net.state_dict().items()}); espera = 0
                else:
                    espera += 1
                    if espera >= paciencia: break
            if melhor[1] is not None: net.load_state_dict(melhor[1])
            net.eval()
            with torch.no_grad():
                zhats.append(net(Xte_s, Xte_a).numpy().ravel())
        zhat = zsc.inverse_transform(np.mean(zhats, axis=0).reshape(-1, 1)).ravel()
        return np.clip(base[te] + zhat, 0.0, _hi_wf(a))
    return fp

print('Framework de DL pronto: archs lstm/gru/attn/tcn | alvos level/diff/residual')


## TCN — convolução causal dilatada (walk-forward)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'TCN (seq+AR)', make_dl_fitpred('tcn', 'level'))
plotar_real_previsto('TCN (seq+AR)')
analisar_residuos('TCN (seq+AR)')

## GRU (walk-forward)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'GRU (seq+AR)', make_dl_fitpred('gru', 'level'))
plotar_real_previsto('GRU (seq+AR)')
analisar_residuos('GRU (seq+AR)')

## LSTM + atenção temporal (walk-forward)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM+Atencao (seq+AR)', make_dl_fitpred('attn', 'level'))
plotar_real_previsto('LSTM+Atencao (seq+AR)')
analisar_residuos('LSTM+Atencao (seq+AR)')

## LSTM differencing — prevê a variação Δ (walk-forward)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM differencing (delta)', make_dl_fitpred('lstm', 'diff'))
plotar_real_previsto('LSTM differencing (delta)')
analisar_residuos('LSTM differencing (delta)')

## LSTM residual — DL sobre o resíduo do baseline ARX (walk-forward)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM residual (base ARX)', make_dl_fitpred('lstm', 'residual'))
plotar_real_previsto('LSTM residual (base ARX)')
analisar_residuos('LSTM residual (base ARX)')

## Sequências alinhadas pelos lags de MI (como no Exp1)

Em vez da janela recente crua (todas as variáveis no lag 0), aqui cada variável é **deslocada pelo
seu lag ótimo de Informação Mútua** (o mesmo `df_tabela_mi` do Exp1) antes de montar a janela —
testando se injetar o conhecimento de lag físico ajuda a DL, como ajudou os modelos do Exp1.

In [ ]:
# --- Sequencias alinhadas pelo lag otimo de cada variavel (MI, como no Exp1) ---
_col_v = 'Variável' if 'Variável' in df_tabela_mi.columns else df_tabela_mi.columns[1]
_lag_map = {r[_col_v]: int(r['Lag (min)']) for _, r in df_tabela_mi.iterrows()}
_df_lag = pd.concat({v: df_dataset[v].shift(_lag_map.get(v, 0)) for v in _feat_proc},
                    axis=1)[_feat_proc].ffill().bfill()
_mat_lag = _df_lag.values.astype(np.float32)
X_seq_lag_wf = np.zeros((N_wf, SEQ_LEN, F), dtype=np.float32)
for i, _pp in enumerate(_pos):
    X_seq_lag_wf[i] = _mat_lag[np.clip(_pp + _off, 0, len(_mat_lag) - 1)]
X_seq_lag_wf = np.nan_to_num(X_seq_lag_wf, nan=0.0)
_psum_lag = X_seq_lag_wf.mean(axis=1)
print(f'Sequencias alinhadas por lag: {X_seq_lag_wf.shape} | lags MI aplicados a {len(_lag_map)} '
      f'variaveis (lag medio {np.mean(list(_lag_map.values())):.0f} min)')

### LSTM (seq+AR) + lags (MI)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM (seq+AR) + lags(MI)', make_dl_fitpred('lstm', 'level', seq=X_seq_lag_wf))
plotar_real_previsto('LSTM (seq+AR) + lags(MI)')
analisar_residuos('LSTM (seq+AR) + lags(MI)')

### LSTM differencing + lags (MI)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM differencing + lags(MI)', make_dl_fitpred('lstm', 'diff', seq=X_seq_lag_wf))
plotar_real_previsto('LSTM differencing + lags(MI)')
analisar_residuos('LSTM differencing + lags(MI)')

### LSTM residual + lags (MI)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM residual + lags(MI)', make_dl_fitpred('lstm', 'residual', seq=X_seq_lag_wf, psum=_psum_lag))
plotar_real_previsto('LSTM residual + lags(MI)')
analisar_residuos('LSTM residual + lags(MI)')

## Veredito: alinhar pelos lags de MI ajuda a DL?

Compara, para cada framing (base / differencing / residual), a versão com janela **crua recente**
contra a versão com as variáveis **alinhadas pelos lags de MI**.

In [ ]:
# --- Sequencias cruas (recentes) vs alinhadas pelos lags de MI ---
import numpy as np
_pares = [('LSTM (seq+AR)', 'LSTM (seq+AR) + lags(MI)'),
          ('LSTM differencing (delta)', 'LSTM differencing + lags(MI)'),
          ('LSTM residual (base ARX)', 'LSTM residual + lags(MI)')]
_d = {r['Modelo']: r for r in RESULTADOS_WF}
_rows = []
for _raw, _lag in _pares:
    if _raw in _d and _lag in _d:
        _r, _l = _d[_raw]['R2 (médio)'], _d[_lag]['R2 (médio)']
        _rows.append({'Framing': _raw, 'R2 cru (recente)': _r, 'R2 +lags(MI)': _l, 'Δ R2': _l - _r})
df_lag_cmp = pd.DataFrame(_rows)
display(df_lag_cmp.style.format({'R2 cru (recente)': '{:.4f}', 'R2 +lags(MI)': '{:.4f}', 'Δ R2': '{:+.4f}'})
        .background_gradient(cmap='RdYlGn', subset=['Δ R2']))
_delta = df_lag_cmp['Δ R2'].mean()
if (df_lag_cmp['Δ R2'] > 0.005).any():
    print(f'>>> Alinhar pelos lags de MI MELHORA a DL em algum framing (Δ medio {_delta:+.4f}).')
else:
    print(f'>>> Alinhar pelos lags de MI NAO melhora a DL (Δ medio {_delta:+.4f}): a janela recente ja carrega a informacao.')

## Residual learning turbinado: ARIMAX real + ensemble de sementes (sem lags)

Duas melhorias sobre o residual learning (que já empatava com o topo), na base **sem lags**:
- **Baseline ARIMAX real** (SARIMAX + exógenas) no lugar do ARX linear — a rede aprende o resíduo do ARIMAX.
- **Ensemble de sementes** (média de 5 redes) para reduzir a variância do fit de DL.
Comparamos as 4 combinações para isolar o efeito de cada uma.

### LSTM residual (ARIMAX real)

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM residual (ARIMAX)', make_dl_fitpred('lstm', 'residual', residual_base='arimax'))
plotar_real_previsto('LSTM residual (ARIMAX)')
analisar_residuos('LSTM residual (ARIMAX)')

### LSTM residual (ARIMAX) + ensemble de 5 sementes

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM residual (ARIMAX)+ens5', make_dl_fitpred('lstm', 'residual', residual_base='arimax', n_seeds=5))
plotar_real_previsto('LSTM residual (ARIMAX)+ens5')
analisar_residuos('LSTM residual (ARIMAX)+ens5')

### LSTM residual (ARX) + ensemble de 5 sementes

In [ ]:
avaliar_wf('Exp3 - LSTM', 'LSTM residual (ARX)+ens5', make_dl_fitpred('lstm', 'residual', residual_base='arx', n_seeds=5))
plotar_real_previsto('LSTM residual (ARX)+ens5')
analisar_residuos('LSTM residual (ARX)+ens5')

## Veredito: ARIMAX real e ensemble arrancam mais R²?

In [ ]:
# --- Compara as variantes de residual learning (base sem lags) ---
import numpy as np
_variantes = ['LSTM residual (base ARX)', 'LSTM residual (ARX)+ens5',
              'LSTM residual (ARIMAX)', 'LSTM residual (ARIMAX)+ens5']
_d = {r['Modelo']: r for r in RESULTADOS_WF}
_rows = [{'Modelo': m, 'R2 (médio)': _d[m]['R2 (médio)'], 'RMSE (médio)': _d[m]['RMSE (médio)']}
         for m in _variantes if m in _d]
df_resid_cmp = pd.DataFrame(_rows).sort_values('R2 (médio)', ascending=False).reset_index(drop=True)
display(df_resid_cmp.style.format({'R2 (médio)': '{:.4f}', 'RMSE (médio)': '{:.4f}'})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)']))
_ref = _d.get('LSTM residual (base ARX)', {}).get('R2 (médio)', float('nan'))
_best = df_resid_cmp.iloc[0]
print(f">>> Melhor: {_best['Modelo']} (R2 {_best['R2 (médio)']:.4f}) | "
      f"ganho vs residual ARX simples: {_best['R2 (médio)'] - _ref:+.4f}")

## Avaliação walk-forward - Experimento 3 (LSTM)

Compara a LSTM com os baselines autorregressivos (persistência e AR(1)) no **mesmo** walk-forward
— este é o veredito sobre o ganho (ou não) de evoluir para Deep Learning.

In [ ]:
# Veredito: LSTM vs baselines de AR (mesmo walk-forward)
import numpy as np
_linhas = [{'Modelo': r['Modelo'], 'R2 (médio)': r['R2 (médio)'], 'RMSE (médio)': r['RMSE (médio)']}
           for r in RESULTADOS_WF if r['Secao'] == 'Exp3 - LSTM']
for _nm, _fn in [('Persistencia (ultimo laudo)', fit_persist), ('AR(1) - Ridge', fit_ar1)]:
    _r2s, _rmses = [], []
    for (a, b) in FOLDS_WF:
        _yt = y_wf.iloc[a:b].values; _pp = _fn(a, b)
        _r2s.append(_r2(_yt, _pp)); _rmses.append(_rmse(_yt, _pp))
    _linhas.append({'Modelo': _nm, 'R2 (médio)': float(np.mean(_r2s)), 'RMSE (médio)': float(np.mean(_rmses))})
df_exp3 = pd.DataFrame(_linhas).sort_values('R2 (médio)', ascending=False).reset_index(drop=True)
display(df_exp3.style.format({'R2 (médio)': '{:.4f}', 'RMSE (médio)': '{:.4f}'})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)']))
_melhor = df_exp3.iloc[0]['Modelo']
_baselines = {'Persistencia (ultimo laudo)', 'AR(1) - Ridge'}
if _melhor not in _baselines:
    print(f'>>> Melhor modelo de DL "{_melhor}" SUPERA os baselines de AR nesta base.')
else:
    print(f'>>> Nenhum modelo de DL supera o baseline "{_melhor}": Deep Learning nao agrega ganho significativo aqui.')


# Experimento 4 - Series temporais: ARIMA e ARIMAX

O CaO Livre e medido em laboratorio de forma **irregular** (intervalo mediano de dezenas de
minutos). Modelos classicos como ARIMA pressupoem amostragem regular, entao aqui tratamos a
**sequencia de medicoes** como a serie temporal (indexada pela ordem da coleta, nao pelo relogio).
Essa e a forma natural de capturar a dinamica do laudo:

- **ARIMA(p,d,q)** modela o CaO Livre a partir dos seus proprios valores passados. Um `ARIMA(1,0,0)`
  com constante e exatamente a *regressao a media do ultimo laudo* (`y(t-1)`), que nas fases
  anteriores se mostrou o sinal mais robusto.
- **ARIMAX** acrescenta variaveis **exogenas** de processo (medias moveis das tags do forno),
  testando se elas agregam informacao **incremental** sobre a estrutura autoregressiva.

Avaliamos com previsao **1-passo-a-frente** (a cada novo laudo, prevemos o proximo) e, ao final,
com **walk-forward** (varias janelas) para um R2 mais honesto que o holdout unico.

## Baselines de referencia (Persistencia e Media)

In [ ]:
# Preparacao da serie de laudos + baselines honestos
# (statsmodels ja consta no requirements; se faltar: pip install statsmodels)
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np, pandas as pd

# Serie indexada pela ORDEM da coleta (uma observacao por laudo real de laboratorio)
serie_lab = df_dataset[target_limpo].dropna()
instantes_lab = serie_lab.index                  # carimbos de tempo reais das coletas
y_serie = serie_lab.reset_index(drop=True)        # serie 0..N-1 para os modelos ARIMA
N = len(y_serie)

# Reaproveita a data de corte definida no calculo de Informacao Mutua (sem vazamento).
# Fallback: 80% das coletas, caso a celula da MI nao tenha sido executada.
try:
    k = int((instantes_lab <= train_cutoff_date).sum())
except NameError:
    k = int(N * 0.8)
k = max(2, min(k, N - 1))

y_tr, y_te = y_serie.iloc[:k], y_serie.iloc[k:]
idx_tr, idx_te = instantes_lab[:k], instantes_lab[k:]
print(f"Coletas totais: {N} | treino: {k} | teste: {N-k}")
print(f"Autocorrelacao lag-1 do laudo: {y_serie.autocorr(1):.3f} "
      f"(quanto maior, mais previsivel pelo passado)")

# --- Baselines de referencia ---
# Persistencia: prever o ultimo laudo conhecido y(t-1)
pers_tr = y_tr.shift(1)
pers_te = y_serie.shift(1).iloc[k:]
m_tr = ~pers_tr.isna()
avaliar_modelo("Baseline Persistencia y(t-1)",
               y_tr[m_tr], pers_tr[m_tr], y_te, pers_te)

# Media do treino (modelo "burro" de referencia para o R2)
media_tr = y_tr.mean()
avaliar_modelo("Baseline Media do treino",
               y_tr, np.full(len(y_tr), media_tr),
               y_te, np.full(len(y_te), media_tr))

In [ ]:
# Validacao walk-forward - baselines (Persistencia, Media, AR(1))
avaliar_wf('Exp4 - Series temporais', 'Persistencia (ultimo laudo)', fit_persist)
avaliar_wf('Exp4 - Series temporais', 'Media do treino', fit_media)
avaliar_wf('Exp4 - Series temporais', 'AR(1) - Ridge', fit_ar1)
plotar_real_previsto('AR(1) - Ridge')

### Analise de residuos

In [ ]:
for _m in ['Persistencia (ultimo laudo)', 'Media do treino', 'AR(1) - Ridge']:
    analisar_residuos(_m)

## ARIMA

In [ ]:
# ARIMA (sem exogenas): selecao de ordem por AIC no TREINO + previsao 1-passo no teste
import numpy as np, pandas as pd

ordens_candidatas = [(1,0,0), (2,0,0), (1,0,1), (2,0,1), (1,1,1), (2,1,1)]

melhor = None
linhas_aic = []
for ordem in ordens_candidatas:
    trend = 'c' if ordem[1] == 0 else 'n'   # constante so faz sentido sem diferenciacao
    try:
        fit = SARIMAX(y_tr, order=ordem, trend=trend,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        linhas_aic.append({'ordem': str(ordem), 'AIC': fit.aic, 'BIC': fit.bic})
        if melhor is None or fit.aic < melhor[1]:
            melhor = (ordem, fit.aic, trend)
    except Exception as e:
        linhas_aic.append({'ordem': str(ordem), 'AIC': np.nan, 'BIC': str(e)[:40]})

display(pd.DataFrame(linhas_aic).sort_values('AIC').reset_index(drop=True))
ordem_arima, _, trend_arima = melhor
print(f"Melhor ordem por AIC: ARIMA{ordem_arima} (trend='{trend_arima}')")

# Reestima os parametros no treino e os aplica a serie completa para gerar
# previsoes 1-passo-a-frente no teste (dynamic=False usa o y real ate t-1 -> sem vazamento)
fit_tr = SARIMAX(y_tr, order=ordem_arima, trend=trend_arima,
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
filtro = SARIMAX(y_serie, order=ordem_arima, trend=trend_arima,
                 enforce_stationarity=False, enforce_invertibility=False).filter(fit_tr.params)

pred_in = filtro.get_prediction(start=0, end=k-1, dynamic=False).predicted_mean
pred_out = filtro.get_prediction(start=k, end=N-1, dynamic=False).predicted_mean

# alinha indices temporais reais para o plot/registro
yt_tr = pd.Series(y_tr.values, index=idx_tr); pt_tr = pd.Series(pred_in.values, index=idx_tr)
yt_te = pd.Series(y_te.values, index=idx_te); pt_te = pd.Series(pred_out.values, index=idx_te)
# descarta a 1a amostra do treino (sem passado para prever)
avaliar_modelo(f"ARIMA{ordem_arima}", yt_tr.iloc[1:], pt_tr.iloc[1:], yt_te, pt_te)
plotar_resultados(yt_tr.iloc[1:], pt_tr.iloc[1:].values, yt_te, pt_te.values, f"ARIMA{ordem_arima}")

In [ ]:
# Validacao walk-forward - ARIMA
if _HAS_SM:
    avaliar_wf('Exp4 - Series temporais', 'ARIMA(2,0,1)', make_arima(None))
    plotar_real_previsto('ARIMA(2,0,1)')

### Analise de residuos

In [ ]:
analisar_residuos('ARIMA(2,0,1)')

## ARIMAX

In [ ]:
# ARIMAX: ARIMA + variaveis exogenas de processo (medias moveis de 180 min)
import numpy as np, pandas as pd

# 1. Exogenas candidatas: media movel de 180 min de cada tag de processo,
#    amostrada nos instantes de laudo. Capta a "condicao media" do forno antes da coleta.
feats_proc = [c for c in df_dataset.columns if c not in [target_col, target_limpo]]
roll = df_dataset[feats_proc].rolling(window=180, min_periods=30).mean()
exog_full = roll.loc[instantes_lab].reset_index(drop=True).ffill().bfill()

# 2. Selecao das exogenas por |correlacao| com o alvo - calculada SOMENTE no treino
corr_tr = exog_full.iloc[:k].corrwith(y_tr).abs().sort_values(ascending=False)
top_exog = corr_tr.head(8).index.tolist()
print("Exogenas selecionadas (corr no treino):")
display(corr_tr.head(8).round(3).to_frame("|corr| treino"))

# 3. Padronizacao ajustada SO no treino (sem vazamento)
exog = exog_full[top_exog]
mu, sd = exog.iloc[:k].mean(), exog.iloc[:k].std().replace(0, 1.0)
exog_z = (exog - mu) / sd
ex_tr, ex_te = exog_z.iloc[:k], exog_z.iloc[k:]

# 4. Ajuste no treino e previsao 1-passo no teste (mesma ordem AR escolhida acima)
fit_x = SARIMAX(y_tr, exog=ex_tr, order=ordem_arima, trend=trend_arima,
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=300)
filtro_x = SARIMAX(y_serie, exog=exog_z, order=ordem_arima, trend=trend_arima,
                   enforce_stationarity=False, enforce_invertibility=False).filter(fit_x.params)

px_in = filtro_x.get_prediction(start=0, end=k-1, exog=ex_tr, dynamic=False).predicted_mean
px_out = filtro_x.get_prediction(start=k, end=N-1, dynamic=False).predicted_mean

yt_tr = pd.Series(y_tr.values, index=idx_tr); ptx_tr = pd.Series(px_in.values, index=idx_tr)
yt_te = pd.Series(y_te.values, index=idx_te); ptx_te = pd.Series(px_out.values, index=idx_te)
avaliar_modelo(f"ARIMAX{ordem_arima} (+processo)", yt_tr.iloc[1:], ptx_tr.iloc[1:], yt_te, ptx_te)
plotar_resultados(yt_tr.iloc[1:], ptx_tr.iloc[1:].values, yt_te, ptx_te.values, f"ARIMAX{ordem_arima}")

In [ ]:
# Validacao walk-forward - ARIMAX (+ processo)
if _HAS_SM:
    avaliar_wf('Exp4 - Series temporais', 'ARIMAX(2,0,1) + processo', make_arima(rep_mean180))
    plotar_real_previsto('ARIMAX(2,0,1) + processo')

### Analise de residuos

In [ ]:
analisar_residuos('ARIMAX(2,0,1) + processo')

## Avaliacao walk-forward - Experimento 4 (Series temporais e baselines)

Inclui as linhas de base (Persistencia / Media / AR(1)) e ARIMA/ARIMAX no **mesmo** protocolo. Grafico Real x Previsto do **melhor R2** desta secao.

In [ ]:
# Consolidacao walk-forward - Experimento 4 (Series temporais e baselines) (compara os modelos da secao, sem recalcular)
df_wf_exp4 = tabela_wf('Exp4 - Series temporais')
print('Comparativo walk-forward - Experimento 4 (Series temporais e baselines):')
display(df_wf_exp4.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df_wf_exp4.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
_melhor = df_wf_exp4.iloc[0]['Modelo']
print('>>> Melhor da secao (R2 médio):', _melhor)
plotar_real_previsto(_melhor, sufixo='  [melhor da secao]')

# Experimento 5 - Modelo otimizado (regularizacao + multi-janela + ensemble)

Os experimentos anteriores mostraram dois problemas: (1) arvores/LSTM **decoram** o treino
(R2 treino ~0,9, teste ~0,1) por excesso de features; (2) o **holdout unico** e instavel,
pois a ultima janela e um regime mais dificil. Esta etapa ataca os dois pontos para ganhar R2
de forma **honesta**:

1. **Backbone autoregressivo**: `y_lag1` e `y_lag2` (ultimos laudos) - o sinal mais forte.
2. **Features de processo multi-janela**: media (60/180/360 min), desvio (180 min), valor
   instantaneo e *slope* (tendencia) de cada tag, amostradas no instante do laudo.
3. **Selecao honesta**: as `k` melhores features por correlacao calculada **so no treino**.
4. **Regularizacao forte + early stopping** (XGB raso, ElasticNet) para nao decorar.
5. **Ensemble** (media de XGB + ElasticNet): combina o nao-linear e o linear, reduzindo variancia.

Avaliamos no **holdout** (para a tabela de ranking) e, principalmente, em **walk-forward**
(5 janelas), que e a metrica confiavel para decidir o melhor modelo.

In [ ]:
# --- Engenharia de features: backbone AR + estatisticas multi-janela de processo ---
import numpy as np, pandas as pd

# Serie de laudos reais (de-plato) e seus instantes
serie_lab_opt = df_dataset[target_limpo].dropna()
inst_opt = serie_lab_opt.index
y_opt = serie_lab_opt.reset_index(drop=True)
N_opt = len(y_opt)

feats_proc = [c for c in df_dataset.columns if c not in [target_col, target_limpo]]

# Estatisticas em multiplas janelas (capturam a "condicao media" e a variabilidade do forno)
mean60  = df_dataset[feats_proc].rolling(60,  min_periods=10).mean()
mean180 = df_dataset[feats_proc].rolling(180, min_periods=30).mean()
mean360 = df_dataset[feats_proc].rolling(360, min_periods=60).mean()
std180  = df_dataset[feats_proc].rolling(180, min_periods=30).std()
# slope: tendencia recente (media dos ultimos 60 min menos a de 120 min atras)
slope60 = mean60 - mean60.shift(120)

X_opt = pd.concat([
    mean60.add_suffix('_m60'), mean180.add_suffix('_m180'), mean360.add_suffix('_m360'),
    std180.add_suffix('_s180'), df_dataset[feats_proc].add_suffix('_now'),
    slope60.add_suffix('_slope'),
], axis=1).loc[inst_opt].reset_index(drop=True)
X_opt = X_opt.ffill().bfill().fillna(0)

# Backbone autoregressivo
ylag1_opt = y_opt.shift(1)
ylag2_opt = y_opt.shift(2)

print(f"Laudos: {N_opt} | features de processo geradas: {X_opt.shape[1]}")

# Corte treino/teste coerente com o resto do notebook (mesma data, sem vazamento)
try:
    k_split = int((inst_opt <= train_cutoff_date).sum())
except NameError:
    k_split = int(N_opt * 0.8)
k_split = max(3, min(k_split, N_opt - 1))
idx_tr_opt, idx_te_opt = inst_opt[:k_split], inst_opt[k_split:]
print(f"Treino: {k_split} | Teste: {N_opt - k_split}")

In [ ]:
# --- Selecao honesta (so no treino) e montagem das matrizes ---
from sklearn.preprocessing import StandardScaler

K_FEATURES = 50  # nº de features de processo mantidas (selecionadas por correlacao no treino)

def selecionar_features(ate_idx, k=K_FEATURES):
    """Top-k features de processo por |correlacao| com o alvo, usando SO o periodo de treino."""
    corr = X_opt.iloc[1:ate_idx].corrwith(y_opt.iloc[1:ate_idx]).abs()
    return corr.sort_values(ascending=False).head(k).index.tolist()

cols_sel = selecionar_features(k_split)
print(f"Selecionadas {len(cols_sel)} features. Top 10:")
display(X_opt.iloc[1:k_split].corrwith(y_opt.iloc[1:k_split]).abs()
        .sort_values(ascending=False).head(10).round(3).to_frame('|corr| treino'))

# Matriz para modelos NAO-lineares (XGB): backbone + processo selecionado
Xfull = pd.concat([ylag1_opt.rename('y_lag1'), ylag2_opt.rename('y_lag2'),
                   X_opt[cols_sel]], axis=1).bfill()
# Matriz para o LINEAR (ElasticNet): mesma base, padronizada
Xlin = pd.concat([ylag1_opt.rename('y_lag1'), X_opt[cols_sel]], axis=1).bfill()

Xfull_tr, Xfull_te = Xfull.iloc[2:k_split], Xfull.iloc[k_split:]
Xlin_tr,  Xlin_te  = Xlin.iloc[2:k_split],  Xlin.iloc[k_split:]
y_tr_opt = y_opt.iloc[2:k_split]
y_te_opt = y_opt.iloc[k_split:]
yt_tr = pd.Series(y_tr_opt.values, index=idx_tr_opt[2:])
yt_te = pd.Series(y_te_opt.values, index=idx_te_opt)

scaler_lin = StandardScaler().fit(Xlin_tr)
Xlin_tr_z = scaler_lin.transform(Xlin_tr)
Xlin_te_z = scaler_lin.transform(Xlin_te)

## Validacao walk-forward por modelo (Exp5)

Cada modelo (ElasticNet, XGBoost, Ensemble) recebe a walk-forward sobre a representacao multi-janela:

In [ ]:
rep_multi = X_opt.reset_index(drop=True)
print('rep_multi (multi-janela):', rep_multi.shape)

## ElasticNet

In [ ]:
# --- Modelo 1: ElasticNet (linear regularizado, interpretavel) ---
from sklearn.linear_model import ElasticNetCV

enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=4, max_iter=5000, random_state=0)
enet.fit(Xlin_tr_z, y_tr_opt)
pred_enet_tr = enet.predict(Xlin_tr_z)
pred_enet_te = enet.predict(Xlin_te_z)

avaliar_modelo("Exp5 ElasticNet (AR + processo)", yt_tr, pred_enet_tr, yt_te, pred_enet_te)
plotar_resultados(yt_tr, pred_enet_tr, yt_te, pred_enet_te, "Exp5 ElasticNet")

In [ ]:
# Validacao walk-forward - Multi-janela + ElasticNet
avaliar_wf('Exp5 - Multi-janela', 'Multi-janela + ElasticNet', make_fitpred(rep_multi, 'enet'))
plotar_real_previsto('Multi-janela + ElasticNet')

### Analise de residuos

In [ ]:
analisar_residuos('Multi-janela + ElasticNet')

## XGBoost

In [ ]:
# --- Modelo 2: XGBoost FORTEMENTE regularizado (raso + subsample + L1/L2) ---
import xgboost as xgb

xgb_reg = xgb.XGBRegressor(
    max_depth=3, n_estimators=300, learning_rate=0.03,
    subsample=0.7, colsample_bytree=0.6,
    reg_lambda=3.0, reg_alpha=0.5, min_child_weight=10,
    random_state=0, n_jobs=-1,
)
xgb_reg.fit(Xfull_tr, y_tr_opt)
pred_xgb_tr = xgb_reg.predict(Xfull_tr)
pred_xgb_te = xgb_reg.predict(Xfull_te)

avaliar_modelo("Exp5 XGBoost regularizado", yt_tr, pred_xgb_tr, yt_te, pred_xgb_te)
plotar_resultados(yt_tr, pred_xgb_tr, yt_te, pred_xgb_te, "Exp5 XGBoost regularizado")

# Importancia das variaveis (quais janelas/tags o modelo mais usa)
display(tabela_importancias(xgb_reg, Xfull.columns, top=15))

In [ ]:
# Validacao walk-forward - Multi-janela + XGBoost
avaliar_wf('Exp5 - Multi-janela', 'Multi-janela + XGBoost', make_fitpred(rep_multi, 'xgb'))
plotar_real_previsto('Multi-janela + XGBoost')

### Analise de residuos

In [ ]:
analisar_residuos('Multi-janela + XGBoost')

## Ensemble

In [ ]:
# --- Modelo 3: ENSEMBLE (media de XGB + ElasticNet) ---
# Combina o nao-linear (XGB) com o linear (ElasticNet): reduz variancia e e o mais
# estavel em walk-forward.
pred_ens_tr = 0.5 * pred_xgb_tr + 0.5 * pred_enet_tr
pred_ens_te = 0.5 * pred_xgb_te + 0.5 * pred_enet_te

avaliar_modelo("Exp5 ENSEMBLE (XGB + ElasticNet)", yt_tr, pred_ens_tr, yt_te, pred_ens_te)
plotar_resultados(yt_tr, pred_ens_tr, yt_te, pred_ens_te, "Exp5 ENSEMBLE")

In [ ]:
# Validacao walk-forward - Multi-janela + Ensemble
avaliar_wf('Exp5 - Multi-janela', 'Multi-janela + Ensemble', fit_ensemble(rep_multi))
plotar_real_previsto('Multi-janela + Ensemble')

### Analise de residuos

In [ ]:
analisar_residuos('Multi-janela + Ensemble')

## Avaliacao walk-forward - Experimento 5 (Multi-janela + ensemble)

Metricas padronizadas, walk-forward e grafico Real x Previsto do **melhor R2** desta secao.

In [ ]:
# Consolidacao walk-forward - Experimento 5 (Multi-janela + ensemble) (compara os modelos da secao, sem recalcular)
df_wf_exp5 = tabela_wf('Exp5 - Multi-janela')
print('Comparativo walk-forward - Experimento 5 (Multi-janela + ensemble):')
display(df_wf_exp5.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df_wf_exp5.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
_melhor = df_wf_exp5.iloc[0]['Modelo']
print('>>> Melhor da secao (R2 médio):', _melhor)
plotar_real_previsto(_melhor, sufixo='')

# Experimento 6 - Variaveis selecionadas (modelo enxuto)

Testa os **mesmos modelos** do Experimento 1, porem usando apenas o **conjunto enxuto** de variaveis
obtido por selecao greedy forward (maximiza o R2 walk-forward). 5 variaveis (+ backbone AR y_lag1/y_lag2)
atingem o mesmo R2 do modelo com todas as 40 - util para um modelo interpretavel e leve.

Conjunto (com leitura fisica p/ cal livre):
- **AR.F1.II12501** - Corrente do motor do forno (torque; intensidade da queima)
- **AR.F1.MFARINHA_FSC_LIMS** - Saturacao de cal (LSF) da farinha produzida (*driver classico*)
- **AR.F1.TI192501** - Temperatura do ar secundario (condicao termica)
- **AR.F1.MCOQUE_PCI_LIMS** - PCI do combustivel (energia de queima)
- **AR.F1.SAIDA_SILO_FARINHA_FSC_LIMS** - LSF da farinha alimentada

Os lags saem do mesmo `df_tabela_mi` (respeita o LAG_MODE escolhido no topo). Edite `VARS_SELECIONADAS`
para testar outros subconjuntos.

In [ ]:
# --- Monta o rep SO com as variaveis selecionadas (mesmos lags de df_tabela_mi) ---
VARS_SELECIONADAS = [
    'AR.F1.II12501',                      # Corrente do motor do forno (torque)
    'AR.F1.MFARINHA_FSC_LIMS',            # Saturacao de cal (LSF) da farinha produzida
    'AR.F1.TI192501',                     # Temperatura do ar secundario
    'AR.F1.MCOQUE_PCI_LIMS',              # PCI do combustivel
    'AR.F1.SAIDA_SILO_FARINHA_FSC_LIMS',  # LSF da farinha alimentada
]
_col_var = 'Variável' if 'Variável' in df_tabela_mi.columns else df_tabela_mi.columns[1]
_lag_de = {r[_col_var]: int(r['Lag (min)']) for _, r in df_tabela_mi.iterrows()}
rep_sel = pd.concat({f'{v}_lag{_lag_de.get(v, 0)}': df_dataset[v].shift(_lag_de.get(v, 0))
                     for v in VARS_SELECIONADAS}, axis=1).loc[inst_wf].reset_index(drop=True).ffill().bfill().fillna(0)
print('rep_sel (variaveis selecionadas):', rep_sel.shape)
print('lags:', {v: _lag_de.get(v, 0) for v in VARS_SELECIONADAS})

## Ridge Regression

In [ ]:
avaliar_wf('Exp6 - Selecionadas', 'Sel + Ridge', make_fitpred(rep_sel, 'ridge'))
plotar_real_previsto('Sel + Ridge')
analisar_residuos('Sel + Ridge')

## PLS (Partial Least Squares)

In [ ]:
avaliar_wf('Exp6 - Selecionadas', 'Sel + PLS', make_fitpred(rep_sel, 'pls'))
plotar_real_previsto('Sel + PLS')
analisar_residuos('Sel + PLS')

## Random Forest

In [ ]:
avaliar_wf('Exp6 - Selecionadas', 'Sel + Random Forest', make_fitpred(rep_sel, 'rf'))
plotar_real_previsto('Sel + Random Forest')
analisar_residuos('Sel + Random Forest')

## Gradient Boosting

In [ ]:
avaliar_wf('Exp6 - Selecionadas', 'Sel + Gradient Boosting', make_fitpred(rep_sel, 'gb'))
plotar_real_previsto('Sel + Gradient Boosting')
analisar_residuos('Sel + Gradient Boosting')

## XGBoost

In [ ]:
avaliar_wf('Exp6 - Selecionadas', 'Sel + XGBoost', make_fitpred(rep_sel, 'xgb'))
plotar_real_previsto('Sel + XGBoost')
analisar_residuos('Sel + XGBoost')

## Regressao Linear Multipla (MLR)

In [ ]:
avaliar_wf('Exp6 - Selecionadas', 'Sel + MLR', make_fitpred(rep_sel, 'mlr'))
plotar_real_previsto('Sel + MLR')
analisar_residuos('Sel + MLR')

## Regressao Polinomial grau 2 (Ridge)

Mesma logica do Ridge do conjunto enxuto, porem expandindo `[AR + variaveis selecionadas]` em termos
polinomiais de **grau 2** (quadrados + interacoes) antes do Ridge. Captura curvatura fisicamente
plausivel da cal livre (resposta nao-linear a LSF/temperatura). Expansao/escalonamento/Ridge ajustados
SO no treino de cada fold (sem vazamento). So faz sentido no conjunto ENXUTO (nas 40 variaveis sobreajusta).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

def make_fitpred_poly(rep, degree=2, k=KSEL_WF):
    """fitpred(a,b) p/ avaliar_wf: [y_lag1,y_lag2 + rep] -> PolynomialFeatures(grau) -> StandardScaler -> RidgeCV.
    Tudo ajustado apenas no treino de cada fold. Clip fisico como os demais."""
    def fp(a, b):
        cols = _sel_wf(rep, a, k)
        X = pd.concat([ylag1_wf.rename('y_lag1'), ylag2_wf.rename('y_lag2'), rep[cols]], axis=1).bfill()
        s0 = 2
        Xtr, ytr, Xte = X.iloc[s0:a].values, y_wf.iloc[s0:a].values, X.iloc[a:b].values
        pf = PolynomialFeatures(degree, include_bias=False)
        Ptr, Pte = pf.fit_transform(Xtr), pf.transform(Xte)
        sc = StandardScaler().fit(Ptr)
        m = RidgeCV(alphas=[.01, .1, 1, 10, 100, 1000]).fit(sc.transform(Ptr), ytr)
        return np.clip(m.predict(sc.transform(Pte)), 0.0, _hi_wf(a))
    return fp

avaliar_wf('Exp6 - Selecionadas', 'Sel + Poly2 (Ridge)', make_fitpred_poly(rep_sel, degree=2))
plotar_real_previsto('Sel + Poly2 (Ridge)')
analisar_residuos('Sel + Poly2 (Ridge)')

## Avaliacao walk-forward - Experimento 6 (Variaveis selecionadas)

In [ ]:
# Consolidacao Exp6 + impacto do enxuto vs completo
df_wf_exp6 = tabela_wf('Exp6 - Selecionadas')
print('Comparativo walk-forward - Experimento 6 (Variaveis selecionadas):')
display(df_wf_exp6.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df_wf_exp6.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
_melhor = df_wf_exp6.iloc[0]['Modelo']
print('>>> Melhor da secao (R2 medio):', _melhor)
plotar_real_previsto(_melhor, sufixo='  [melhor da secao]')

# Impacto: enxuto ({N} vars) vs completo (Exp1, todas as variaveis)
_d = {r['Modelo']: r['R2 (médio)'] for r in RESULTADOS_WF}
print(f'\n--- Impacto (enxuto = {len(VARS_SELECIONADAS)} vars vs completo = todas) ---')
for _s, _f in [('Sel + Ridge', 'Lags + Ridge'), ('Sel + PLS', 'Lags + PLS'),
               ('Sel + XGBoost', 'Lags + XGBoost'), ('Sel + MLR', 'Lags + MLR')]:
    if _s in _d and _f in _d:
        print(f'  {_s:<22} {_d[_s]:+.3f}   vs   {_f:<20} {_d[_f]:+.3f}   (delta {_d[_s]-_d[_f]:+.3f})')

# Experimento 7 - Conjunto enxuto + Deep Learning

Testa os modelos de Deep Learning (mesma infraestrutura do Experimento 3) usando SOMENTE o conjunto
enxuto `VARS_SELECIONADAS` (5 variaveis), para verificar se o DL agrega ganho sobre o linear enxuto
(Exp6 `Sel + Ridge`) e como se compara ao DL completo (Exp3). Reaproveita `make_dl_fitpred` passando
uma sequencia 3D so com as variaveis selecionadas.

In [ ]:
# --- Sequencias 3D SO com as variaveis selecionadas (mesma janela/subamostragem do Exp3) ---
_mat_sel = df_dataset[VARS_SELECIONADAS].ffill().bfill().values.astype(np.float32)
X_seq_sel_wf = np.zeros((N_wf, SEQ_LEN, len(VARS_SELECIONADAS)), dtype=np.float32)
for i, _p in enumerate(_pos):
    X_seq_sel_wf[i] = _mat_sel[np.clip(_p + _off, 0, len(_mat_sel) - 1)]
X_seq_sel_wf = np.nan_to_num(X_seq_sel_wf, nan=0.0)
_psum_sel = X_seq_sel_wf.mean(axis=1)        # medias de janela (exogenas do baseline ARX enxuto)
print('X_seq_sel_wf:', X_seq_sel_wf.shape, '| variaveis:', VARS_SELECIONADAS)

## LSTM (seq+AR)

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel LSTM (seq+AR)', make_dl_fitpred('lstm', 'level', seq=X_seq_sel_wf))
plotar_real_previsto('Sel LSTM (seq+AR)')
analisar_residuos('Sel LSTM (seq+AR)')

## GRU

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel GRU', make_dl_fitpred('gru', 'level', seq=X_seq_sel_wf))
plotar_real_previsto('Sel GRU')
analisar_residuos('Sel GRU')

## TCN

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel TCN', make_dl_fitpred('tcn', 'level', seq=X_seq_sel_wf))
plotar_real_previsto('Sel TCN')
analisar_residuos('Sel TCN')

## LSTM + atencao temporal

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel LSTM+Atencao', make_dl_fitpred('attn', 'level', seq=X_seq_sel_wf))
plotar_real_previsto('Sel LSTM+Atencao')
analisar_residuos('Sel LSTM+Atencao')

## LSTM differencing (delta)

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel LSTM differencing', make_dl_fitpred('lstm', 'diff', seq=X_seq_sel_wf))
plotar_real_previsto('Sel LSTM differencing')
analisar_residuos('Sel LSTM differencing')

## LSTM residual (base ARX)

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel LSTM residual (ARX)', make_dl_fitpred('lstm', 'residual', seq=X_seq_sel_wf, psum=_psum_sel))
plotar_real_previsto('Sel LSTM residual (ARX)')
analisar_residuos('Sel LSTM residual (ARX)')

## LSTM residual (ARX) + ensemble de 5 sementes

In [ ]:
avaliar_wf('Exp7 - Sel + DL', 'Sel LSTM residual (ARX)+ens5', make_dl_fitpred('lstm', 'residual', seq=X_seq_sel_wf, psum=_psum_sel, n_seeds=5))
plotar_real_previsto('Sel LSTM residual (ARX)+ens5')
analisar_residuos('Sel LSTM residual (ARX)+ens5')

## Avaliacao walk-forward - Experimento 7 (Conjunto enxuto + Deep Learning)

In [ ]:
# Consolidacao Exp7 + veredito (DL enxuto vs linear enxuto vs DL completo)
df_wf_exp7 = tabela_wf('Exp7 - Sel + DL')
print('Comparativo walk-forward - Experimento 7 (Conjunto enxuto + Deep Learning):')
display(df_wf_exp7.drop(columns=['Secao']).style.format({c: '{:.3f}' for c in df_wf_exp7.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))
_melhor = df_wf_exp7.iloc[0]['Modelo']
print('>>> Melhor DL enxuto (R2 medio):', _melhor)
plotar_real_previsto(_melhor, sufixo='  [melhor da secao]')

_d = {r['Modelo']: r['R2 (médio)'] for r in RESULTADOS_WF}
_exp3 = [r for r in RESULTADOS_WF if r['Secao'] == 'Exp3 - LSTM']
_best_full = max(_exp3, key=lambda r: r['R2 (médio)']) if _exp3 else None
print('\n--- Veredito: ha ganho com DL no conjunto enxuto? ---')
if 'Sel + Ridge' in _d:
    print(f"  Linear enxuto (Exp6 Sel + Ridge):     {_d['Sel + Ridge']:+.3f}")
print(f"  Melhor DL enxuto (Exp7): {_melhor} = {df_wf_exp7.iloc[0]['R2 (médio)']:+.3f}")
if _best_full is not None:
    print(f"  Melhor DL completo (Exp3): {_best_full['Modelo']} = {_best_full['R2 (médio)']:+.3f}")

# Comparativo final dos experimentos

Tabela comparativa de **todos os modelos**, avaliados no **mesmo** protocolo walk-forward (agregada de todas as seções). Base de decisão para escolher melhor modelo.

In [ ]:
# Comparativo final padronizado (walk-forward agregado de todas as secoes)
df_comparativo = tabela_wf()
display(df_comparativo.style.format({c: '{:.3f}' for c in df_comparativo.columns if c not in ['Secao','Modelo']})
        .background_gradient(cmap='RdYlGn', subset=['R2 (médio)','R2 (global)'])
        .background_gradient(cmap='RdYlGn_r', subset=['RMSE (médio)','MAE (médio)']))

# Ranking visual por R2 médio
fig, ax = plt.subplots(figsize=(12, max(4, 0.5*len(df_comparativo))))
d = df_comparativo.sort_values('R2 (médio)')
cores = ['#C0504D' if v < 0 else '#1F3864' for v in d['R2 (médio)']]
ax.barh(d['Modelo'], d['R2 (médio)'], color=cores); ax.axvline(0, color='grey', lw=0.8)
ax.set_xlabel('R2 médio (walk-forward, 5 janelas) - quanto maior, melhor')
ax.set_title('Comparativo final padronizado - todos os modelos'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

melhor_geral = df_comparativo.iloc[0]['Modelo']
print('>>> Melhor modelo geral (R2 médio):', melhor_geral)
plotar_real_previsto(melhor_geral, sufixo='')

# Nota metodologica - split aleatorio vs walk-forward

Trabalhos da literatura (ex.: dissertacao UFMG 2019, sensor virtual de cal livre) reportam R2 ~0,72
usando **split treino/teste ALEATORIO 80/20**. Este notebook usa **validacao walk-forward temporal**
(honesta para um sensor que roda em tempo real prevendo o futuro), obtendo R2 ~0,25.

A celula abaixo demonstra, de forma reproduzivel, **por que aquele numero nao se transfere**: avaliar
o MESMO modelo (Ridge, mesmas features) com split aleatorio **infla o R2** por vazamento temporal - a
serie do alvo e autocorrelacionada e usamos o ultimo laudo (`y_lag`) como feature, entao no split
aleatorio o vizinho temporal do ponto de teste cai no treino (a resposta "vaza").

In [ ]:
# Split ALEATORIO (como a literatura) vs WALK-FORWARD temporal (honesto), mesmo modelo Ridge
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV

def _ridge_pred(Xtr, ytr, Xte):
    sc = StandardScaler().fit(Xtr)
    m = RidgeCV(alphas=[.1, 1, 10, 100]).fit(sc.transform(Xtr), ytr)
    return np.clip(m.predict(sc.transform(Xte)), 0.0, 1.5 * ytr.max())

def _r2_walkforward(X, yv):
    return float(np.mean([_r2(yv[a:b], _ridge_pred(X[2:a], yv[2:a], X[a:b])) for (a, b) in FOLDS_WF]))

def _r2_aleatorio(X, yv, seeds=20, frac=0.8):
    idx = np.arange(2, len(yv)); r2s = []
    for s in range(seeds):
        perm = np.random.RandomState(s).permutation(idx); k = int(len(perm) * frac)
        r2s.append(_r2(yv[perm[k:]], _ridge_pred(X[perm[:k]], yv[perm[:k]], X[perm[k:]])))
    return float(np.mean(r2s))

yv = y_wf.values
_linhas = []
for _nome, _rep in [('Enxuto (AR + 5 selecionadas)', rep_sel), ('Completo (AR + lags MI)', rep_lags)]:
    X = pd.concat([ylag1_wf.rename('y_lag1'), ylag2_wf.rename('y_lag2'), _rep], axis=1).bfill().fillna(0).values
    wf, al = _r2_walkforward(X, yv), _r2_aleatorio(X, yv)
    _linhas.append({'Conjunto': _nome, 'R2 walk-forward (honesto)': wf,
                    'R2 split aleatorio (literatura)': al, 'Inflacao do split': al - wf})
df_split = pd.DataFrame(_linhas)
display(df_split.style.format({c: '{:+.3f}' for c in df_split.columns if c != 'Conjunto'})
        .background_gradient(cmap='Reds', subset=['Inflacao do split']))
print('>>> O split ALEATORIO infla o R2 (vazamento temporal: y_lag + autocorrelacao do alvo).')
print('>>> Para um sensor em tempo real, a validacao honesta e a WALK-FORWARD temporal (~0,25).')
print('>>> Por isso o ~0,72 da literatura (split aleatorio) nao representa o desempenho real de campo.')